# Projeto Final - Ecossistema de Dados


---


| Campo | Preencha |
|-------|----------|
| **Grupo** | 1|
| **Integrantes** |Rafael Borges, Alvaro Koene, Bruno Mainardi, Fernando Henrique, Rafaela Eduarda, Kaique Geska|
| **Dataset escolhido** |Campeonato Brasileiro de Futebol |
| **Fonte do dataset** |https://www.kaggle.com/datasets/adaoduque/campeonato-brasileiro-de-futebol?select=campeonato-brasileiro-gols.csv |
| **Data de entrega** | 17/04/2026 |


## Perguntas de negócio que este projeto responde

> Defina aqui as 5 perguntas de negócio que seu DW e Data Lake vão responder.  
> Estas perguntas guiarão toda a modelagem e as queries analíticas da Seção 6.

1. *(Vantagens de jogar em casa)*
2. *(Desempenho Histórico dos times)*
3. *('Times que fez mais gols')*
4. *(times mais violentos)*
5. *(Estatisticas Jogadores)*

---

## Decisão arquitetural

**Por que ETL para o Data Warehouse?**  
*(escreva aqui)*

**Por que ELT para o Data Lake?**  
*(escreva aqui)*

**Banco de dados escolhido para o DW e justificativa:**  
*(escreva aqui)*


---
## Estrutura do notebook

| Seção | Conteúdo |
|-------|----------|
| **Setup** | Instalação, imports, pastas |
| **Extract** | Leitura e exploração do dataset |
| **Raw/Staging** | Salvar bruto com metadados |
| **ETL → DW** | Qualidade → Dimensões → Fato → DuckDB |
| **ELT → Data Lake** | Bronze → Silver (SQL) → Gold (SQL) |
| **Monitoramento** | Log de pipeline + validação |
| **Analytics** | 5 queries OLAP respondendo as perguntas |
| **Conclusão** | Comparação ETL vs. ELT + recomendação |


---
# Seção 0: Setup


In [ ]:
# 0.1 - Instale as dependências
# Instale: duckdb, pandas, pyarrow
# Adicione outras que seu dataset exigir
#Abrir o Terminal e executar:
#pip install duckdb pandas pyarrow

In [ ]:
# 0.2 - Imports e configurações globais
import pandas as pd
import numpy as np
import duckdb
import json
import datetime
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# 0.3 - Criar estrutura de pastas do projeto
#
# Estrutura esperada:
#   Projeto Final/
#   ├── data/                  ← coloque o dataset aqui
#   ├── datalake/
#   │   ├── bronze/
#   │   ├── silver/
#   │   └── gold/
#   ├── logs/
#   └── dw_projeto.duckdb      ← criado na Seção 3

for pasta in ['data', 'datalake/bronze', 'datalake/silver', 'datalake/gold', 'logs']:
    Path(pasta).mkdir(parents=True, exist_ok=True)

print('Estrutura de pastas criada.')

Estrutura de pastas criada.


In [ ]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("adaoduque/campeonato-brasileiro-de-futebol")

os.makedirs("data", exist_ok=True)

for file in os.listdir(path):
    src = os.path.join(path, file)
    dst = os.path.join("data", file)
    shutil.copy(src, dst)

print("Arquivos copiados para ./data")

100%|██████████| 691k/691k [00:00<00:00, 31.4MB/s]

Extracting files...
Arquivos copiados para ./data


### Função de Monitoramento

Esta função já está implementada. **Você deve chamá-la em toda etapa do pipeline.**  
Ela registra: nome da etapa, status, quantidade de registros antes e depois, e timestamp.

** Ajuste ao projeto, se for necessário


In [ ]:
# 0.4: Função de monitoramento do pipeline (NÃO altere esta célula)

PIPELINE_LOG = []

def log_etapa(etapa: str, status: str,
              qtd_antes: int = None, qtd_depois: int = None,
              obs: str = '') -> None:
    """
    Registra o resultado de uma etapa do pipeline.

    Parâmetros
    ----------
    etapa     : nome descritivo da etapa  (ex: 'ETL - Remover duplicatas')
    status    : 'OK' ou 'FALHA'
    qtd_antes : número de registros antes da operação
    qtd_depois: número de registros após a operação
    obs       : observação livre
    """
    removidos = (qtd_antes or 0) - (qtd_depois or 0) if qtd_antes and qtd_depois else None
    entrada = {
        'etapa'     : etapa,
        'status'    : status,
        'qtd_antes' : qtd_antes,
        'qtd_depois': qtd_depois,
        'removidos' : removidos,
        'timestamp' : datetime.datetime.now().isoformat(),
        'obs'       : obs
    }
    PIPELINE_LOG.append(entrada)
    qtd_str = f'{qtd_depois:,}' if qtd_depois is not None else '-'
    rem_str = f'  ({removidos:+,} registros)' if removidos else ''
    print(f"[{status:^5}]  {etapa:<45}  {qtd_str} registros{rem_str}")


def exibir_log() -> pd.DataFrame:
    """Exibe o log completo do pipeline como DataFrame."""
    if not PIPELINE_LOG:
        print('Nenhuma etapa registrada ainda.')
        return None
    return pd.DataFrame(PIPELINE_LOG)


def salvar_log(caminho: str = 'logs/pipeline_log.json') -> None:
    """Salva o log em JSON para auditoria."""
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(PIPELINE_LOG, f, ensure_ascii=False, indent=2)
    print(f'Log salvo em: {caminho}')


print('Função log_etapa disponível.')

Função log_etapa disponível.


---
# Seção 1: Extract

> **Objetivo:** carregar o dataset e entender o que você tem antes de qualquer transformação.


In [ ]:
# 1.1 - Leia o dataset
#
# Dica: se houver problema de encoding, tente encoding='latin-1' ou encoding='windows-1252'
# Dica: se o separador for ponto-e-vírgula, use sep=';'
#
# Ao final, registre no log:
#   log_etapa('Extract - Leitura do dataset', 'OK', qtd_depois=len(df))

df_gols = pd.read_csv('data/campeonato-brasileiro-gols.csv')
df_cartoes = pd.read_csv('data/campeonato-brasileiro-cartoes.csv')
df_estatisticas = pd.read_csv('data/campeonato-brasileiro-estatisticas-full.csv')
df_full = pd.read_csv('data/campeonato-brasileiro-full.csv')

log_etapa('Extract - Leitura do dataset - (gols)', 'OK', qtd_depois=len(df_gols))
log_etapa('Extract - Leitura do dataset - (cartoes)', 'OK', qtd_depois=len(df_cartoes))
log_etapa('Extract - Leitura do dataset - (estatisticas)', 'OK', qtd_depois=len(df_estatisticas))
log_etapa('Extract - Leitura do dataset - (full)', 'OK', qtd_depois=len(df_full))



[ OK  ]  Extract - Leitura do dataset - (gols)          10,820 registros
[ OK  ]  Extract - Leitura do dataset - (cartoes)       20,953 registros
[ OK  ]  Extract - Leitura do dataset - (estatisticas)  18,330 registros
[ OK  ]  Extract - Leitura do dataset - (full)          9,165 registros


In [ ]:
# 1.2 - Exploração inicial
#
# Exiba:
#   a) Shape (linhas × colunas)
#   b) Tipos de dados de cada coluna (dtypes)
#   c) Contagem de nulos por coluna
#   d) Estatísticas descritivas das colunas numéricas
#   e) As 5 primeiras linhas
print("-"*8 +"DF Gols"+"-"*8 )
print(df_gols.shape)
print(df_gols.dtypes)
print(df_gols.isnull().sum())
print(df_gols.describe())
print(df_gols.head())

print("\n"+"-"*8+"DF cartoes"+"-"*8)
print(df_cartoes.shape)
print(df_cartoes.dtypes)
print(df_cartoes.isnull().sum())
print(df_cartoes.describe())
print(df_cartoes.head())

print("\n"+"-"*8+"DF estatisticas"+"-"*8)
print(df_estatisticas.shape)
print(df_estatisticas.dtypes)
print(df_estatisticas.isnull().sum())
print(df_estatisticas.describe())
print(df_estatisticas.head())

print("\n"+"-"*8+"DF full"+"-"*8)
print(df_full.shape)
print(df_full.dtypes)
print(df_full.isnull().sum())
print(df_full.describe())
print(df_full.head())


--------DF Gols--------
(10820, 6)
partida_id      int64
rodata          int64
clube          object
atleta         object
minuto         object
tipo_de_gol    object
dtype: object
partida_id        0
rodata            0
clube             0
atleta            0
minuto            0
tipo_de_gol    9527
dtype: int64
       partida_id   rodata
count    10820.00 10820.00
mean      6915.88    19.64
std       1323.99    11.04
min       4607.00     1.00
25%       5757.00    10.00
50%       6939.00    20.00
75%       8070.00    29.00
max       9165.00    38.00
   partida_id  rodata          clube                   atleta minuto  \
0        4607       1     Fluminense             Rafael Sóbis     31   
1        4607       1     Fluminense                     Fred     45   
2        4607       1     Fluminense  Nirley da Silva Fonseca     59   
3        4608       1  Internacional         Charles Aránguiz      6   
4        4612       1       Cruzeiro           Marcelo Moreno     90   

  tipo_de_

In [ ]:
# 1.3 - Diagnóstico de qualidade
#
# Investigue e documente:
#   a) Quantidade de linhas duplicadas
#   b) Colunas com mais de 10% de nulos
#   c) Colunas de data que estão como string
#   d) Valores inválidos em colunas numéricas (negativos onde não deveria)
#   e) Inconsistências de padronização (ex: estados em maiúsculo e minúsculo)
#
# Este diagnóstico guiará o tratamento na Seção 3
print("-"*8 +"DF Gols"+"-"*8 )
print("Quantidade de linhas duplicadas")
print(df_gols.duplicated().sum())
print("\nColunas com mais de 10% de nulos")
print((df_gols.isnull().sum() / len(df_gols)) * 100)
print("\nColunas de data que estão como string")
print(df_gols.select_dtypes(include=['object']).nunique())
print("\nValores inválidos em colunas numéricas (negativos onde não deveria)")
print(df_gols.select_dtypes(include=['int64', 'float64']).min())
print("\nInconsistências de padronização (ex: estados em maiúsculo e minúsculo)")
print(df_gols.select_dtypes(include=['int64', 'float64']).max())

print("\n"+"-"*8+"DF cartoes"+"-"*8)
print("Quantidade de linhas duplicadas")
print(df_cartoes.duplicated().sum())
print("\nColunas com mais de 10% de nulos")
print((df_cartoes.isnull().sum() / len(df_cartoes)) * 100)
print("\nColunas de data que estão como string")
print(df_cartoes.select_dtypes(include=['object']).nunique())
print("\nValores inválidos em colunas numéricas (negativos onde não deveria)")
print(df_cartoes.select_dtypes(include=['int64', 'float64']).min())
print("\nInconsistências de padronização (ex: estados em maiúsculo e minúsculo)")
print(df_cartoes.select_dtypes(include=['int64', 'float64']).max())

print("\n"+"-"*8+"DF estatisticas"+"-"*8)
print("Quantidade de linhas duplicadas")
print(df_estatisticas.duplicated().sum())
print("\nColunas com mais de 10% de nulos")
print((df_estatisticas.isnull().sum() / len(df_estatisticas)) * 100)
print("\nColunas de data que estão como string")
print(df_estatisticas.select_dtypes(include=['object']).nunique())
print("\nValores inválidos em colunas numéricas (negativos onde não deveria)")
print(df_estatisticas.select_dtypes(include=['int64', 'float64']).min())
print("\nInconsistências de padronização (ex: estados em maiúsculo e minúsculo)")
print(df_estatisticas.select_dtypes(include=['int64', 'float64']).max())

print("\n"+"-"*8+"DF full"+"-"*8)
print("Quantidade de linhas duplicadas")
print(df_full.duplicated().sum())
print("\nColunas com mais de 10% de nulos")
print((df_full.isnull().sum() / len(df_full)) * 100)
print("\nColunas de data que estão como string")
print(df_full.select_dtypes(include=['object']).nunique())
print("\nValores inválidos em colunas numéricas (negativos onde não deveria)")
print(df_full.select_dtypes(include=['int64', 'float64']).min())
print("\nInconsistências de padronização (ex: estados em maiúsculo e minúsculo)")
print(df_full.select_dtypes(include=['int64', 'float64']).max())

--------DF Gols--------
Quantidade de linhas duplicadas
0

Colunas com mais de 10% de nulos
partida_id     0.00
rodata         0.00
clube          0.00
atleta         0.00
minuto         0.00
tipo_de_gol   88.05
dtype: float64

Colunas de data que estão como string
clube            35
atleta         1636
minuto          118
tipo_de_gol       2
dtype: int64

Valores inválidos em colunas numéricas (negativos onde não deveria)
partida_id    4607
rodata           1
dtype: int64

Inconsistências de padronização (ex: estados em maiúsculo e minúsculo)
partida_id    9165
rodata          38
dtype: int64

--------DF cartoes--------
Quantidade de linhas duplicadas
0

Colunas com mais de 10% de nulos
partida_id   0.00
rodata       0.00
clube        0.00
cartao       0.00
atleta       0.03
num_camisa   1.84
posicao      5.72
minuto       0.00
dtype: float64

Colunas de data que estão como string
clube        34
cartao        2
atleta     2290
posicao       5
minuto      122
dtype: int64

Valores in

---
# Seção 2: Staging

> **Regra fundamental:** a Staging recebe o dado **bruto**, exatamente como veio da fonte.  
> Nenhuma transformação é aplicada aqui, apenas metadados de ingestão são adicionados.
>
> Por que isso importa? Se o pipeline falhar depois, você pode reprocessar a partir do staging  
> sem precisar voltar à fonte original.


In [ ]:
# 2.1 - Salvar o dataset bruto como Parquet no staging
#
# Antes de salvar, adicione as colunas de metadados:
#   _source_file  : nome do arquivo de origem (string)
#   _ingested_at  : timestamp da ingestão (pd.Timestamp.now())
#   _batch_id     : identificador da execução, use datetime.date.today().isoformat()
#
# Salvar em: datalake/bronze/staging_raw.parquet
#
# Ao final:
#   log_etapa('Staging - Salvar bruto', 'OK', qtd_antes=len(df), qtd_depois=len(df_staging))

# Dicionário organizando os 4 dataframes e seus nomes de arquivo
datasets = {
    'gols': (df_gols, 'campeonato-brasileiro-gols.csv'),
    'cartoes': (df_cartoes, 'campeonato-brasileiro-cartoes.csv'),
    'estatisticas': (df_estatisticas, 'campeonato-brasileiro-estatisticas-full.csv'),
    'full': (df_full, 'campeonato-brasileiro-full.csv')
}

# Laço para salvar e etiquetar os 4 arquivos
for nome, (df_original, arquivo_origem) in datasets.items():
    df_staging = df_original.copy()

    # Adicionar colunas de metadados
    df_staging['_source_file'] = arquivo_origem
    df_staging['_ingested_at'] = pd.Timestamp.now()
    df_staging['_batch_id'] = datetime.date.today().isoformat()

    # Salvar em Parquet na camada bronze
    caminho_staging = f'datalake/bronze/staging_{nome}.parquet'
    df_staging.to_parquet(caminho_staging, index=False)

    # Registrar no log
    log_etapa(f'Staging - Salvar bruto ({nome})', 'OK', qtd_antes=len(df_original), qtd_depois=len(df_staging))


[ OK  ]  Staging - Salvar bruto (gols)                  10,820 registros
[ OK  ]  Staging - Salvar bruto (cartoes)               20,953 registros
[ OK  ]  Staging - Salvar bruto (estatisticas)          18,330 registros
[ OK  ]  Staging - Salvar bruto (full)                  9,165 registros


In [ ]:
# 2.2: Verificar o arquivo de staging

# Leia o Parquet de volta e confirme:
#   a) O número de linhas bate com o dataset original
#   b) As colunas de metadados estão presentes
#   c) Nenhuma transformação foi aplicada (os dados são idênticos ao CSV)


metadados = ['_source_file', '_ingested_at', '_batch_id']

# Laço para verificar os 4 arquivos salvos
for nome, (df_original, _) in datasets.items():
    caminho_staging = f'datalake/bronze/staging_{nome}.parquet'
    df_verificacao = pd.read_parquet(caminho_staging)

    linhas_batem = len(df_original) == len(df_verificacao)
    metadados_presentes = all(col in df_verificacao.columns for col in metadados)
    dados_intactos = df_original.equals(df_verificacao.drop(columns=metadados))

    print(f"--- Verificando: {nome} ---")
    print(f"A) Linhas conferem? {'Sim' if linhas_batem else 'Não'} (Original: {len(df_original)} | Staging: {len(df_verificacao)})")
    print(f"B) Metadados presentes? {'Sim' if metadados_presentes else 'Não'}")
    print(f"C) Dados base idênticos ao original? {'Sim' if dados_intactos else 'Não'}\n")


--- Verificando: gols ---
A) Linhas conferem? Sim (Original: 10820 | Staging: 10820)
B) Metadados presentes? Sim
C) Dados base idênticos ao original? Sim

--- Verificando: cartoes ---
A) Linhas conferem? Sim (Original: 20953 | Staging: 20953)
B) Metadados presentes? Sim
C) Dados base idênticos ao original? Sim

--- Verificando: estatisticas ---
A) Linhas conferem? Sim (Original: 18330 | Staging: 18330)
B) Metadados presentes? Sim
C) Dados base idênticos ao original? Sim

--- Verificando: full ---
A) Linhas conferem? Sim (Original: 9165 | Staging: 9165)
B) Metadados presentes? Sim
C) Dados base idênticos ao original? Sim



---
# Seção 3: ETL → Data Warehouse

## Fluxo
```
Staging  ──►  TRANSFORM (Python)  ──►  LOAD  ──►  dw_projeto.duckdb
```

> Toda a lógica de negócio acontece em Python, **antes** de qualquer dado ir para o banco.  
> O DW recebe apenas DataFrames já limpos e modelados.

## Seu modelo Star Schema

*(Substitua o diagrama abaixo pelo modelo que seu grupo projetou)*

```
                       dim_data
                          │
 dim_tipo_gol  ────  fato_eventos  ────  dim_clube
                    │           │
                dim_atleta     dim_cartao
```

| Tabela | Colunas | Origem |
|--------|---------|--------|
| `dim_data` | sk_data, data, ano, trimestre, mes, dia_semana, is_weekend | coluna de data |
| `dim_clube` | sk_clube, clube | nomes dos clubes presentes nos eventos |
| `dim_atleta` | sk_atleta, atleta | nome dos atletas presentes n
| `fato_eventos` | sk_fato, sk_data, sk_cartao, sk_ | todas |


## 3.1: EXTRACT: Tratamento de Qualidade

> Trabalhe sobre o DataFrame lido do staging (não do CSV original).  
> Documente cada decisão: o que foi removido e por quê.


In [ ]:
# 3.1a - Carregar o staging para trabalhar
#
# Leia datalake/bronze/staging_raw.parquet
# Trabalhe sempre a partir daqui - não do CSV original
df_partidas = pd.read_parquet('datalake/bronze/staging_full.parquet')
df_gols = pd.read_parquet('datalake/bronze/staging_gols.parquet')
df_cartoes = pd.read_parquet('datalake/bronze/staging_cartoes.parquet')
df_estatisticas = pd.read_parquet('datalake/bronze/staging_estatisticas.parquet')

In [ ]:
df_partidas.info()
df_partidas.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9165 entries, 0 to 9164
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID                  9165 non-null   int64         
 1   rodata              9165 non-null   int64         
 2   data                9165 non-null   object        
 3   hora                9165 non-null   object        
 4   mandante            9165 non-null   object        
 5   visitante           9165 non-null   object        
 6   formacao_mandante   4190 non-null   object        
 7   formacao_visitante  4190 non-null   object        
 8   tecnico_mandante    4555 non-null   object        
 9   tecnico_visitante   4555 non-null   object        
 10  vencedor            9165 non-null   object        
 11  arena               9165 non-null   object        
 12  mandante_Placar     9165 non-null   int64         
 13  visitante_Placar    9165 non-null   int64       

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado,arrecadacao,_source_file,_ingested_at,_batch_id
0,1,1,29/03/2003,16:00,Guarani,Vasco,None,None,None,None,Guarani,Brinco de Ouro,4,2,SP,RJ,NaN,campeonato-brasileiro-full.csv,2026-04-13 23:47:37.821465,2026-04-13
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,None,None,None,None,Athletico-PR,Arena da Baixada,2,0,PR,RS,NaN,campeonato-brasileiro-full.csv,2026-04-13 23:47:37.821465,2026-04-13
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,None,None,None,None,-,Maracanã,1,1,RJ,PR,NaN,campeonato-brasileiro-full.csv,2026-04-13 23:47:37.821465,2026-04-13
3,4,1,30/03/2003,16:00,Goias,Paysandu,None,None,None,None,-,Serra Dourada,2,2,GO,PA,NaN,campeonato-brasileiro-full.csv,2026-04-13 23:47:37.821465,2026-04-13
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,None,None,None,None,-,Beira Rio,1,1,RS,SP,NaN,campeonato-brasileiro-full.csv,2026-04-13 23:47:37.821465,2026-04-13


In [ ]:
df_partidas = df_partidas.rename(columns={'rodata': 'rodada'})

In [ ]:
#arrumando data
df_partidas['data'] = pd.to_datetime(
    df_partidas['data'],
    dayfirst=True,
    errors='coerce'
)

In [ ]:
print(df_partidas['data'].min(), df_partidas['data'].max())
print("Nulos:", df_partidas['data'].isna().sum())

2003-03-29 00:00:00 2025-12-07 00:00:00
Nulos: 0


In [ ]:
#filtrando os dados de 2014 para usar
antes = len(df_partidas)

df_partidas_2014 = df_partidas[
    df_partidas['data'].dt.year >= 2014
].copy()

depois = len(df_partidas_2014)

print("Antes:", antes)
print("Depois:", depois)

Antes: 9165
Depois: 4559


In [ ]:
print(df_partidas_2014['data'].min(), df_partidas_2014['data'].max())

2014-04-19 00:00:00 2025-12-07 00:00:00


In [ ]:
ids_validos = df_partidas_2014['ID'].astype(int).unique()

print("Partidas válidas:", len(ids_validos))

Partidas válidas: 4559


In [ ]:
log_etapa(
    'ETL - Filtrar partidas (2014+)',
    'OK',
    qtd_antes=antes,
    qtd_depois=depois
)

[ OK  ]  ETL - Filtrar partidas (2014+)                 4,559 registros  (+4,606 registros)


In [ ]:
#filtrando dados 2014 em gols
antes_gols = len(df_gols)

df_gols = df_gols[df_gols['partida_id'].isin(ids_validos)].copy()

depois_gols = len(df_gols)

print("Gols - antes:", antes_gols)
print("Gols - depois:", depois_gols)

log_etapa(
    'ETL - Filtrar gols (2014+)',
    'OK',
    qtd_antes=antes_gols,
    qtd_depois=depois_gols
)

Gols - antes: 10820
Gols - depois: 10820
[ OK  ]  ETL - Filtrar gols (2014+)                     10,820 registros


In [ ]:
#filtrando dados 2014 em cartoes
antes_cartoes = len(df_cartoes)

df_cartoes = df_cartoes[df_cartoes['partida_id'].isin(ids_validos)].copy()

depois_cartoes = len(df_cartoes)

print("Cartões - antes:", antes_cartoes)
print("Cartões - depois:", depois_cartoes)

log_etapa(
    'ETL - Filtrar cartoes (2014+)',
    'OK',
    qtd_antes=antes_cartoes,
    qtd_depois=depois_cartoes
)

Cartões - antes: 20953
Cartões - depois: 20953
[ OK  ]  ETL - Filtrar cartoes (2014+)                  20,953 registros


In [ ]:
#filtrando dados 2014 em estatisticas
antes_estat = len(df_estatisticas)

df_estatisticas = df_estatisticas[df_estatisticas['partida_id'].isin(ids_validos)].copy()

depois_estat = len(df_estatisticas)

print("Estatísticas - antes:", antes_estat)
print("Estatísticas - depois:", depois_estat)

log_etapa(
    'ETL - Filtrar estatisticas (2014+)',
    'OK',
    qtd_antes=antes_estat,
    qtd_depois=depois_estat
)

Estatísticas - antes: 18330
Estatísticas - depois: 9118
[ OK  ]  ETL - Filtrar estatisticas (2014+)             9,118 registros  (+9,212 registros)


In [ ]:
df_partidas.columns

Index(['ID', 'rodada', 'data', 'hora', 'mandante', 'visitante',
       'formacao_mandante', 'formacao_visitante', 'tecnico_mandante',
       'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar',
       'visitante_Placar', 'mandante_Estado', 'visitante_Estado',
       'arrecadacao', '_source_file', '_ingested_at', '_batch_id'],
      dtype='object')

In [ ]:
df_estatisticas.columns

Index(['partida_id', 'rodata', 'clube', 'chutes', 'chutes_no_alvo',
       'posse_de_bola', 'passes', 'precisao_passes', 'faltas',
       'cartao_amarelo', 'cartao_vermelho', 'impedimentos', 'escanteios',
       '_source_file', '_ingested_at', '_batch_id'],
      dtype='object')

In [ ]:
df_cartoes.columns

Index(['partida_id', 'rodata', 'clube', 'cartao', 'atleta', 'num_camisa',
       'posicao', 'minuto', '_source_file', '_ingested_at', '_batch_id'],
      dtype='object')

In [ ]:
df_gols.columns

Index(['partida_id', 'rodata', 'clube', 'atleta', 'minuto', 'tipo_de_gol',
       '_source_file', '_ingested_at', '_batch_id'],
      dtype='object')

In [ ]:
# 3.1b - Remover duplicatas
#
# Identifique as colunas que definem uma linha única
# Exiba a quantidade antes e depois
#
# log_etapa('ETL - Remover duplicatas', 'OK', qtd_antes=..., qtd_depois=...)
print("PARTIDAS")
print(df_partidas.duplicated().sum())
print(df_partidas.duplicated(subset=['ID']).sum())

print("\nGOLS")
print(df_gols.duplicated().sum())
print(df_gols.duplicated(subset=['partida_id', 'clube', 'atleta', 'minuto', 'tipo_de_gol']).sum())

print("\nCARTOES")
print(df_cartoes.duplicated().sum())
print(df_cartoes.duplicated(subset=['partida_id', 'clube', 'atleta', 'cartao', 'minuto']).sum())

print("\nESTATISTICAS")
print(df_estatisticas.duplicated().sum())
print(df_estatisticas.duplicated(subset=['partida_id', 'clube']).sum())

PARTIDAS
0
0

GOLS
0
0

CARTOES
0
0

ESTATISTICAS
0
0


In [ ]:
# 3.1c - Converter tipos de dados
#
# Converta colunas de data para datetime
# Converta colunas numéricas que estejam como string
#
# log_etapa('ETL - Converter tipos', 'OK', qtd_depois=len(df_clean))
print("PARTIDAS")
print(df_partidas.dtypes)

print("\nGOLS")
print(df_gols.dtypes)

print("\nCARTOES")
print(df_cartoes.dtypes)

print("\nESTATISTICAS")
print(df_estatisticas.dtypes)

PARTIDAS
ID                             int64
rodada                         int64
data                  datetime64[ns]
hora                          object
mandante                      object
visitante                     object
formacao_mandante             object
formacao_visitante            object
tecnico_mandante              object
tecnico_visitante             object
vencedor                      object
arena                         object
mandante_Placar                int64
visitante_Placar               int64
mandante_Estado               object
visitante_Estado              object
arrecadacao                  float64
_source_file                  object
_ingested_at          datetime64[us]
_batch_id                     object
dtype: object

GOLS
partida_id               int64
rodata                   int64
clube                   object
atleta                  object
minuto                  object
tipo_de_gol             object
_source_file            object
_ingested_at  

In [ ]:
# 3.1d - Tratar nulos e valores inválidos
#
# Para cada coluna com problema, decida e justifique:
#   - Remover a linha?
#   - Preencher com valor padrão / mediana / moda?
#   - Manter como está e tratar na query?
#
# log_etapa('ETL - Tratar nulos', 'OK', qtd_antes=..., qtd_depois=...)
df_partidas.isnull().sum()

,0
ID,0
rodada,0
data,0
hora,0
mandante,0
visitante,0
formacao_mandante,4975
formacao_visitante,4975
tecnico_mandante,4610
tecnico_visitante,4610


In [ ]:
df_cartoes.isnull().sum()

,0
partida_id,0
rodata,0
clube,0
cartao,0
atleta,6
num_camisa,386
posicao,1198
minuto,0
_source_file,0
_ingested_at,0


In [ ]:
df_gols.isnull().sum()

,0
partida_id,0
rodata,0
clube,0
atleta,0
minuto,0
tipo_de_gol,9527
_source_file,0
_ingested_at,0
_batch_id,0


In [ ]:
df_estatisticas.isnull().sum()

,0
partida_id,0
rodata,0
clube,0
chutes,0
chutes_no_alvo,0
posse_de_bola,1550
passes,0
precisao_passes,3854
faltas,0
cartao_amarelo,0


In [ ]:
df_gols['tipo_de_gol'] = df_gols['tipo_de_gol'].fillna('Normal')

In [ ]:
df_cartoes = df_cartoes.dropna(subset=['atleta'])

In [ ]:
df_partidas = df_partidas.drop(columns=['arrecadacao'])

In [ ]:
df_partidas = df_partidas.drop(columns=[
    'formacao_mandante',
    'formacao_visitante'
])

In [ ]:
df_partidas = df_partidas.drop(columns=[
    'tecnico_mandante',
    'tecnico_visitante'
])

In [ ]:
# 3.1e - Padronizações (se necessário)
#
# Exemplos: estados em minúsculo → maiúsculo, strip de espaços,
# categorias com nomes inconsistentes, etc.
#for df in [df_partidas, df_gols, df_cartoes, df_estatisticas]:
# log_etapa('ETL - Padronizações', 'OK', qtd_depois=len(df_clean))
for df in [df_partidas, df_gols, df_cartoes, df_estatisticas]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

In [ ]:
df_partidas['mandante'] = df_partidas['mandante'].str.title()
df_partidas['visitante'] = df_partidas['visitante'].str.title()

df_gols['clube'] = df_gols['clube'].str.title()
df_cartoes['clube'] = df_cartoes['clube'].str.title()
df_estatisticas['clube'] = df_estatisticas['clube'].str.title()

In [ ]:
df_gols['atleta'] = df_gols['atleta'].str.title()
df_cartoes['atleta'] = df_cartoes['atleta'].str.title()

In [ ]:
df_partidas['mandante_Estado'] = df_partidas['mandante_Estado'].str.upper()
df_partidas['visitante_Estado'] = df_partidas['visitante_Estado'].str.upper()

In [ ]:
df_gols['tipo_de_gol'] = df_gols['tipo_de_gol'].str.title()

In [ ]:
df_cartoes['cartao'] = df_cartoes['cartao'].str.title()

## 3.2: TRANSFORM: Construção das Dimensões

> Para cada dimensão: extraia as colunas relevantes, elimine duplicatas  
> e adicione uma **Surrogate Key (SK)** sequencial começando em 1.


In [ ]:
# 3.2a - Construir dim_data
#
# Colunas obrigatórias: sk_data, data, ano, trimestre, mes, nome_mes,
#                       dia_semana, nome_dia, is_weekend
#
# Dica: use pd.to_datetime e os acessores .dt.year, .dt.quarter, etc.
#
# log_etapa('ETL - Construir dim_data', 'OK', qtd_depois=len(dim_data))
dim_data = df_partidas_2014[['data']].drop_duplicates().copy()
dim_data = dim_data.sort_values('data').reset_index(drop=True)
dim_data['sk_data'] = dim_data.index + 1

In [ ]:
dim_data['data'] = pd.to_datetime(dim_data['data'], errors='coerce')
dim_data['ano'] = dim_data['data'].dt.year
dim_data['trimestre'] = dim_data['data'].dt.quarter
dim_data['mes'] = dim_data['data'].dt.month
dim_data['nome_mes'] = dim_data['data'].dt.month_name()
dim_data['dia_semana'] = dim_data['data'].dt.weekday  # 0 = segunda
dim_data['nome_dia'] = dim_data['data'].dt.day_name()
dim_data['is_weekend'] = dim_data['dia_semana'].isin([5, 6])

In [ ]:
dim_data = dim_data[
    [
        'sk_data',
        'data',
        'ano',
        'trimestre',
        'mes',
        'nome_mes',
        'dia_semana',
        'nome_dia',
        'is_weekend'
    ]
]

In [ ]:
log_etapa(
    'ETL - Construir dim_data',
    'OK',
    qtd_depois=len(dim_data)
)

[ OK  ]  ETL - Construir dim_data                       1,231 registros


In [ ]:
# 3.2b - Construir segunda dimensão (defina o nome)
#
# Exemplo: dim_cliente, dim_produto, dim_local, dim_vendedor...
#
# Extraia as colunas relevantes do df_clean
# Elimine duplicatas (.drop_duplicates())
# Gere SK sequencial (.reset_index(drop=True); dim['sk_X'] = dim.index + 1)
#
# log_etapa('ETL - Construir dim_...', 'OK', qtd_depois=len(dim_X))
clubes = pd.concat([
    df_partidas['mandante'],
    df_partidas['visitante'],
    df_gols['clube'],
    df_cartoes['clube'],
    df_estatisticas['clube']
], ignore_index=True)

In [ ]:
dim_clube = pd.DataFrame({'clube': clubes.dropna().unique()})

In [ ]:
dim_clube['sk_clube'] = dim_clube.index + 1

In [ ]:
dim_clube = dim_clube[['sk_clube', 'clube']]

In [ ]:
log_etapa(
    'ETL - Construir dim_clube',
    'OK',
    qtd_depois=len(dim_clube)
)

[ OK  ]  ETL - Construir dim_clube                      46 registros


In [ ]:
# 3.2c - Construir terceira dimensão (se houver)
#
# log_etapa('ETL - Construir dim_...', 'OK', qtd_depois=len(dim_Y))
atletas = pd.concat([
    df_gols['atleta'],
    df_cartoes['atleta']
], ignore_index=True)

In [ ]:
dim_atleta = pd.DataFrame({'atleta': atletas.dropna().unique()})

In [ ]:
dim_atleta['sk_atleta'] = dim_atleta.index + 1

In [ ]:
dim_atleta = dim_atleta[['sk_atleta', 'atleta']]
dim_atleta.head()
print(dim_atleta.shape)

(2457, 2)


In [ ]:
log_etapa(
    'ETL - Construir dim_atleta',
    'OK',
    qtd_depois=len(dim_atleta)
)

[ OK  ]  ETL - Construir dim_atleta                     2,457 registros


In [ ]:
# 3.2d - Construir demais dimensões (adicione células conforme necessário)
dim_tipo_gol = pd.DataFrame({
    'tipo_de_gol': df_gols['tipo_de_gol'].dropna().unique()
})

In [ ]:
dim_tipo_gol['sk_tipo_gol'] = dim_tipo_gol.index + 1

In [ ]:
dim_tipo_gol = dim_tipo_gol[['sk_tipo_gol', 'tipo_de_gol']]

In [ ]:
log_etapa(
    'ETL - Construir dim_tipo_gol',
    'OK',
    qtd_depois=len(dim_tipo_gol)
)

[ OK  ]  ETL - Construir dim_tipo_gol                   3 registros


In [ ]:
df_cartoes['cartao'] = df_cartoes['cartao'].str.title()

dim_cartao = pd.DataFrame({
    'cartao': df_cartoes['cartao'].dropna().unique()
})

In [ ]:
dim_cartao['sk_cartao'] = dim_cartao.index + 1

In [ ]:
log_etapa(
    'ETL - Construir dim_cartao',
    'OK',
    qtd_depois=len(dim_cartao)
)

[ OK  ]  ETL - Construir dim_cartao                     2 registros


In [ ]:
# 3.2e - Construir a tabela fato
#
# log_etapa('ETL - Construir fato', 'OK', qtd_depois=len(fato))
map_clube = dict(zip(dim_clube['clube'], dim_clube['sk_clube']))

In [ ]:
map_atleta = dict(zip(dim_atleta['atleta'], dim_atleta['sk_atleta']))

In [ ]:
map_tipo_gol = dict(zip(dim_tipo_gol['tipo_de_gol'], dim_tipo_gol['sk_tipo_gol']))

In [ ]:
map_partida_data = dict(zip(df_partidas['ID'], df_partidas['data']))
map_data_sk = dict(zip(dim_data['data'], dim_data['sk_data']))

In [ ]:
map_cartao = dict(zip(dim_cartao['cartao'], dim_cartao['sk_cartao']))

In [ ]:
fato_gols = df_gols.copy()
fato_gols['data_str'] = fato_gols['partida_id'].map(map_partida_data)
fato_gols['data_dt'] = pd.to_datetime(fato_gols['data_str'], dayfirst=True, errors='coerce')
fato_gols['sk_data'] = fato_gols['data_dt'].map(map_data_sk)
fato_gols['sk_clube'] = fato_gols['clube'].map(map_clube)
fato_gols['sk_atleta'] = fato_gols['atleta'].map(map_atleta)
fato_gols['sk_tipo_gol'] = fato_gols['tipo_de_gol'].map(map_tipo_gol)

fato_gols['sk_cartao'] = None
fato_gols['tipo_evento'] = 'GOL'
fato_gols['qtd_gol'] = 1
fato_gols['qtd_cartao'] = 0

fato_gols = fato_gols.drop(columns=['clube', 'atleta', 'tipo_de_gol', 'rodata', 'data_str', 'data_dt'])

fato_gols = fato_gols[[
    'partida_id',
    'sk_data',
    'sk_clube',
    'sk_atleta',
    'sk_tipo_gol',
    'sk_cartao',
    'tipo_evento',
    'minuto',
    'qtd_gol',
    'qtd_cartao'
]]

In [ ]:
fato_cartoes = df_cartoes.copy()
fato_cartoes['data_str'] = fato_cartoes['partida_id'].map(map_partida_data)
fato_cartoes['data_dt'] = pd.to_datetime(fato_cartoes['data_str'], dayfirst=True, errors='coerce')
fato_cartoes['sk_data'] = fato_cartoes['data_dt'].map(map_data_sk)
fato_cartoes['sk_clube'] = fato_cartoes['clube'].map(map_clube)
fato_cartoes['sk_atleta'] = fato_cartoes['atleta'].map(map_atleta)
fato_cartoes['sk_cartao'] = fato_cartoes['cartao'].map(map_cartao)

fato_cartoes['sk_tipo_gol'] = None
fato_cartoes['tipo_evento'] = 'CARTAO'
fato_cartoes['qtd_gol'] = 0
fato_cartoes['qtd_cartao'] = 1

fato_cartoes = fato_cartoes.drop(columns=['clube', 'atleta', 'cartao', 'rodata', 'data_str', 'data_dt'])

fato_cartoes = fato_cartoes[[
    'partida_id',
    'sk_data',
    'sk_clube',
    'sk_atleta',
    'sk_tipo_gol',
    'sk_cartao',
    'tipo_evento',
    'minuto',
    'qtd_gol',
    'qtd_cartao'
]]

In [ ]:
print(fato_gols.columns.tolist())
print(fato_cartoes.columns.tolist())

['partida_id', 'sk_data', 'sk_clube', 'sk_atleta', 'sk_tipo_gol', 'sk_cartao', 'tipo_evento', 'minuto', 'qtd_gol', 'qtd_cartao']
['partida_id', 'sk_data', 'sk_clube', 'sk_atleta', 'sk_tipo_gol', 'sk_cartao', 'tipo_evento', 'minuto', 'qtd_gol', 'qtd_cartao']


In [ ]:
fato_eventos = pd.concat([fato_gols, fato_cartoes], ignore_index=True)

fato_eventos['qtd_gol'] = fato_eventos['qtd_gol'].fillna(0).astype(int)
fato_eventos['qtd_cartao'] = fato_eventos['qtd_cartao'].fillna(0).astype(int)

In [ ]:
print(fato_eventos.head())
print(fato_eventos.columns)
print(fato_eventos[['tipo_evento', 'qtd_gol', 'qtd_cartao']].sample(10))
print(fato_eventos['qtd_cartao'].sum())
print(fato_eventos[fato_eventos['qtd_cartao'] > 0].head())


   partida_id  sk_data  sk_clube  sk_atleta sk_tipo_gol sk_cartao tipo_evento  \
0        4607        1        13          1           1      None         GOL   
1        4607        1        13          2           2      None         GOL   
2        4607        1        13          3           3      None         GOL   
3        4608        1         5          4           1      None         GOL   
4        4612        2         9          5           1      None         GOL   

  minuto  qtd_gol  qtd_cartao  
0     31        1           0  
1     45        1           0  
2     59        1           0  
3      6        1           0  
4     90        1           0  
Index(['partida_id', 'sk_data', 'sk_clube', 'sk_atleta', 'sk_tipo_gol',
       'sk_cartao', 'tipo_evento', 'minuto', 'qtd_gol', 'qtd_cartao'],
      dtype='object')
      tipo_evento  qtd_gol  qtd_cartao
5554          GOL        1           0
27191      CARTAO        0           1
21246      CARTAO        0           1


In [ ]:
log_etapa(
    'ETL - Construir fato_eventos',
    'OK',
    qtd_depois=len(fato_eventos)
)

[ OK  ]  ETL - Construir fato_eventos                   31,767 registros


## 3.3: LOAD: Criar o DW e carregar as tabelas

> **Ordem obrigatória:** dimensões primeiro, fato por último.  
> A fato contém FKs que referenciam as dimensões, carregar na ordem errada quebra a integridade.


In [ ]:
import duckdb

conn = duckdb.connect('dw_projeto.duckdb')


conn.execute("DROP TABLE IF EXISTS fato_eventos;")
conn.execute("DROP TABLE IF EXISTS dim_data;")
conn.execute("DROP TABLE IF EXISTS dim_clube;")
conn.execute("DROP TABLE IF EXISTS dim_atleta;")
conn.execute("DROP TABLE IF EXISTS dim_tipo_gol;")
conn.execute("DROP TABLE IF EXISTS dim_cartao;")

# Dimensão data
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_data (
    sk_data INTEGER PRIMARY KEY,
    data DATE,
    ano INTEGER,
    trimestre INTEGER,
    mes INTEGER,
    nome_mes VARCHAR,
    dia_semana INTEGER,
    nome_dia VARCHAR,
    is_weekend BOOLEAN
)
""")

# Dimensão clube
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_clube (
    sk_clube INTEGER PRIMARY KEY,
    clube VARCHAR
)
""")

# Dimensão atleta
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_atleta (
    sk_atleta INTEGER PRIMARY KEY,
    atleta VARCHAR
)
""")

# Dimensão tipo de gol
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_tipo_gol (
    sk_tipo_gol INTEGER PRIMARY KEY,
    tipo_de_gol VARCHAR
)
""")

# Dimensão tipo de cartão
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_cartao (
    sk_cartao INTEGER PRIMARY KEY,
    cartao VARCHAR
)
""")

# Tabela fato
conn.execute("""
CREATE TABLE IF NOT EXISTS fato_eventos (
    partida_id INTEGER,
    sk_data INTEGER,
    sk_clube INTEGER,
    sk_atleta INTEGER,
    sk_tipo_gol INTEGER,
    sk_cartao INTEGER,
    tipo_evento VARCHAR,
    minuto VARCHAR,
    qtd_gol INTEGER,
    qtd_cartao INTEGER,
    FOREIGN KEY (sk_data) REFERENCES dim_data(sk_data),
    FOREIGN KEY (sk_clube) REFERENCES dim_clube(sk_clube),
    FOREIGN KEY (sk_atleta) REFERENCES dim_atleta(sk_atleta),
    FOREIGN KEY (sk_tipo_gol) REFERENCES dim_tipo_gol(sk_tipo_gol),
    FOREIGN KEY (sk_cartao) REFERENCES dim_cartao(sk_cartao)
)
""")

In [ ]:
# 3.3b - Carregar os DataFrames no DW

# Registrar os DataFrames pandas como views temporárias no DuckDB
conn.register('vw_dim_data', dim_data)
conn.register('vw_dim_clube', dim_clube)
conn.register('vw_dim_atleta', dim_atleta)
conn.register('vw_dim_tipo_gol', dim_tipo_gol)
conn.register('vw_dim_cartao', dim_cartao)
conn.register('vw_fato_eventos', fato_eventos)

# Carregar dimensões primeiro
conn.execute("""
    INSERT INTO dim_data (
        sk_data, data, ano, trimestre, mes, nome_mes, dia_semana, nome_dia, is_weekend
    )
    SELECT
        sk_data, data, ano, trimestre, mes, nome_mes, dia_semana, nome_dia, is_weekend
    FROM vw_dim_data
""")
log_etapa('DW - Carregar dim_data', 'OK', qtd_depois=len(dim_data))

conn.execute("""
    INSERT INTO dim_clube (sk_clube, clube)
    SELECT sk_clube, clube
    FROM vw_dim_clube
""")
log_etapa('DW - Carregar dim_clube', 'OK', qtd_depois=len(dim_clube))

conn.execute("""
    INSERT INTO dim_atleta (sk_atleta, atleta)
    SELECT sk_atleta, atleta
    FROM vw_dim_atleta
""")
log_etapa('DW - Carregar dim_atleta', 'OK', qtd_depois=len(dim_atleta))

conn.execute("""
    INSERT INTO dim_tipo_gol (sk_tipo_gol, tipo_de_gol)
    SELECT sk_tipo_gol, tipo_de_gol
    FROM vw_dim_tipo_gol
""")
log_etapa('DW - Carregar dim_tipo_gol', 'OK', qtd_depois=len(dim_tipo_gol))

conn.execute("""
    INSERT INTO dim_cartao (sk_cartao, cartao)
    SELECT sk_cartao, cartao
    FROM vw_dim_cartao
""")
log_etapa('DW - Carregar dim_cartao', 'OK', qtd_depois=len(dim_cartao))

# Carregar fato por último
conn.execute("""
    INSERT INTO fato_eventos (
        partida_id,
        sk_data,
        sk_clube,
        sk_atleta,
        sk_tipo_gol,
        sk_cartao,
        tipo_evento,
        minuto,
        qtd_gol,
        qtd_cartao
    )
    SELECT
        partida_id,
        sk_data,
        sk_clube,
        sk_atleta,
        sk_tipo_gol,
        sk_cartao,
        tipo_evento,
        minuto,
        qtd_gol,
        qtd_cartao
    FROM vw_fato_eventos
""")
log_etapa('DW - Carregar fato_eventos', 'OK', qtd_depois=len(fato_eventos))

[ OK  ]  DW - Carregar dim_data                         1,231 registros
[ OK  ]  DW - Carregar dim_clube                        46 registros
[ OK  ]  DW - Carregar dim_atleta                       2,457 registros
[ OK  ]  DW - Carregar dim_tipo_gol                     3 registros
[ OK  ]  DW - Carregar dim_cartao                       2 registros
[ OK  ]  DW - Carregar fato_eventos                     31,767 registros


In [ ]:
# 3.3c - Validar o DW
#
# Para cada tabela, execute: SELECT COUNT(*) FROM tabela
# Verifique:
#   a) A contagem bate com os DataFrames?
#   b) A soma das métricas da fato bate com o total do dataset original?
#   c) Existem SKs nulas na fato?
#
# log_etapa('DW - Validação pós-carga', 'OK', qtd_depois=total_fato)
print("dim_data:", conn.execute("SELECT COUNT(*) FROM dim_data").fetchone()[0])
print("dim_clube:", conn.execute("SELECT COUNT(*) FROM dim_clube").fetchone()[0])
print("dim_atleta:", conn.execute("SELECT COUNT(*) FROM dim_atleta").fetchone()[0])
print("dim_tipo_gol:", conn.execute("SELECT COUNT(*) FROM dim_tipo_gol").fetchone()[0])
print("dim_cartao:", conn.execute("SELECT COUNT(*) FROM dim_cartao").fetchone()[0])
print("fato_eventos:", conn.execute("SELECT COUNT(*) FROM fato_eventos").fetchone()[0])

dim_data: 1231
dim_clube: 46
dim_atleta: 2457
dim_tipo_gol: 3
dim_cartao: 2
fato_eventos: 31767


In [ ]:
fato_eventos[fato_eventos['tipo_evento'] == 'CARTAO'][['sk_data', 'sk_clube', 'sk_atleta', 'sk_cartao']].isnull().sum()

,0
sk_data,0
sk_clube,0
sk_atleta,0
sk_cartao,0


---
# Seção 4: ELT → Data Lake (Arquitetura Medallion)

## Fluxo
```
Bronze ──► Silver ──► Gold
```

> Os mesmos dados da Seção 3, mas desta vez o dado é carregado bruto primeiro.  
> As transformações acontecem em SQL, dentro do Data Lake.  
> Python serve apenas para orquestrar, toda lógica está nas queries.


## 4.1: Camada Bronze (dado bruto)

> Bronze é um **espelho fiel da fonte**.  
> Nenhum dado é alterado, removido ou filtrado.  
> É idêntico ao staging, a diferença é que agora está na estrutura do Data Lake.


In [ ]:
# 4.1 - Salvar o dado bruto na camada Bronze

  # A camada Bronze já foi construída na Seção 2 (Staging),
  # portanto nesta etapa utilizamos diretamente esses arquivos.

# Leia o staging_raw.parquet e salve em datalake/bronze/bronze_dataset.parquet

conn = duckdb.connect('dw_projeto.duckdb')

conn.execute(""" COPY(Select * from '/content/datalake/bronze/staging_gols.parquet') TO '/content/datalake/bronze/bronze_gols.parquet'""")
conn.execute(""" COPY(Select * from '/content/datalake/bronze/staging_cartoes.parquet') TO '/content/datalake/bronze/bronze_cartoes.parquet'""")
conn.execute(""" COPY(Select * from '/content/datalake/bronze/staging_estatisticas.parquet') TO '/content/datalake/bronze/bronze_estatisticas.parquet'""")
conn.execute(""" COPY(Select * from '/content/datalake/bronze/staging_full.parquet') TO '/content/datalake/bronze/bronze_full.parquet'""")

# O arquivo Bronze deve ser idêntico ao staging
# Confirme: exiba o número de linhas e colunas

#Consultando quantidade de Linhas staging
QtdLinhasStaging_gols          = conn.execute("""Select count(*) from '/content/datalake/bronze/staging_gols.parquet'""").fetchone()[0]
QtdLinhasStaging_cartoes       = conn.execute("""Select count(*) from '/content/datalake/bronze/staging_cartoes.parquet'""").fetchone()[0]
QtdLinhasStaging_estatisticas  = conn.execute("""Select count(*) from '/content/datalake/bronze/staging_estatisticas.parquet'""").fetchone()[0]
QtdLinhasStaging_full          = conn.execute("""Select count(*) from '/content/datalake/bronze/staging_full.parquet'""").fetchone()[0]

#Consultando quantidade de Linhas bronze
QtdLinhasBronze_gols           = conn.execute("""Select count(*) from '/content/datalake/bronze/bronze_gols.parquet'""").fetchone()[0]
QtdLinhasBronze_cartoes        = conn.execute("""Select count(*) from '/content/datalake/bronze/bronze_cartoes.parquet'""").fetchone()[0]
QtdLinhasBronze_estatisticas   = conn.execute("""Select count(*) from '/content/datalake/bronze/bronze_estatisticas.parquet'""").fetchone()[0]
QtdLinhasBronze_full           = conn.execute("""Select count(*) from '/content/datalake/bronze/bronze_full.parquet'""").fetchone()[0]

#Consultando quantidade de Colunas staging
QtdColunasStaging_gols         = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/staging_gols.parquet'""").df()
QtdColunasStaging_cartoes      = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/staging_cartoes.parquet'""").df()
QtdColunasStaging_estatisticas = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/staging_estatisticas.parquet'""").df()
QtdColunasStaging_full         = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/staging_full.parquet'""").df()

#Consultando quantidade de Colunas bronze
QtdColunasBronze_gols          = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/bronze_gols.parquet'""").df()
QtdColunasBronze_cartoes       = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/bronze_cartoes.parquet'""").df()
QtdColunasBronze_estatisticas  = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/bronze_estatisticas.parquet'""").df()
QtdColunasBronze_full          = conn.execute(""" DESCRIBE SELECT * FROM '/content/datalake/bronze/bronze_full.parquet'""").df()

print("Quantidade de linhas Staging_gols",         QtdLinhasStaging_gols)
print("Quantidade de linhas Bronze_gols",          QtdLinhasBronze_gols)
print("Quantidade de colunas Staging_gols",        len(QtdColunasStaging_gols))
print("Quantidade de colunas Bronze_gols",         len(QtdColunasBronze_gols))

print("\nQuantidade de linhas Staging_cartoes",      QtdLinhasStaging_cartoes)
print("Quantidade de linhas Bronze_cartoes",        QtdLinhasBronze_cartoes)
print("Quantidade de colunas Staging_cartoes",      len(QtdColunasStaging_cartoes))
print("Quantidade de colunas Bronze_cartoes",       len(QtdColunasBronze_cartoes))

print("\nQuantidade de linhas Staging_estatisticas", QtdLinhasStaging_estatisticas)
print("Quantidade de linhas Bronze_estatisticas",   QtdLinhasBronze_estatisticas)
print("Quantidade de colunas Staging_estatisticas", len(QtdColunasStaging_estatisticas))
print("Quantidade de colunas Bronze_estatisticas",  len(QtdColunasBronze_estatisticas))

print("\nQuantidade de linhas Staging_full",         QtdLinhasStaging_full)
print("Quantidade de linhas Bronze_full",           QtdLinhasBronze_full)
print("Quantidade de colunas Staging_full",         len(QtdColunasStaging_full))
print("Quantidade de colunas Bronze_full",          len(QtdColunasBronze_full))

# log_etapa('ELT - Bronze (LOAD)', 'OK', qtd_depois=...)

log_etapa('ELT - Ler, salvar e comparar staging com bronze gols',         'OK', QtdLinhasStaging_gols,         QtdLinhasBronze_gols)
log_etapa('ELT - Ler, salvar e comparar staging com bronze cartões',      'OK', QtdLinhasStaging_cartoes,      QtdLinhasBronze_cartoes)
log_etapa('ELT - Ler, salvar e comparar staging com bronze estatisticas', 'OK', QtdLinhasStaging_estatisticas, QtdLinhasBronze_estatisticas)
log_etapa('ELT - Ler, salvar e comparar staging com bronze full',         'OK', QtdLinhasStaging_full,         QtdLinhasBronze_full)


Quantidade de linhas Staging_gols 10820
Quantidade de linhas Bronze_gols 10820
Quantidade de colunas Staging_gols 9
Quantidade de colunas Bronze_gols 9

Quantidade de linhas Staging_cartoes 20953
Quantidade de linhas Bronze_cartoes 20953
Quantidade de colunas Staging_cartoes 11
Quantidade de colunas Bronze_cartoes 11

Quantidade de linhas Staging_estatisticas 18330
Quantidade de linhas Bronze_estatisticas 18330
Quantidade de colunas Staging_estatisticas 16
Quantidade de colunas Bronze_estatisticas 16

Quantidade de linhas Staging_full 9165
Quantidade de linhas Bronze_full 9165
Quantidade de colunas Staging_full 20
Quantidade de colunas Bronze_full 20
[ OK  ]  ELT - Ler, salvar e comparar staging com bronze gols  10,820 registros
[ OK  ]  ELT - Ler, salvar e comparar staging com bronze cartões  20,953 registros
[ OK  ]  ELT - Ler, salvar e comparar staging com bronze estatisticas  18,330 registros
[ OK  ]  ELT - Ler, salvar e comparar staging com bronze full  9,165 registros


:## 4.2: Camada Silver (limpeza com SQL)

> Na Silver, use **apenas SQL** sobre o arquivo Bronze.  
> O DuckDB lê Parquet diretamente: `read_parquet('datalake/bronze/bronze_dataset.parquet')`  
> Para salvar: `COPY (SELECT ...) TO 'datalake/silver/silver.parquet' (FORMAT PARQUET)`


In [ ]:
# 4.2a - Conectar ao DuckDB para as transformações SQL
#
# Use uma conexão separada (ou a mesma do DW - DuckDB suporta múltiplas)
# conn_lake = duckdb.connect()  # in-memory é suficiente para o Data Lake

conn = duckdb.connect('dw_projeto.duckdb')

from pathlib import Path
Path('/content/datalake/silver').mkdir(parents=True, exist_ok=True)
print('Conexão estabelecida.')

Conexão estabelecida.


In [ ]:
# 4.2b - Construir a Silver com SQL
#
# Escreva uma única query que:
#   1. Leia o Bronze: FROM read_parquet('datalake/bronze/bronze_dataset.parquet')
#   2. Renomeie colunas com espaços usando aliases ("Coluna X" AS coluna_x)
#   3. Converta colunas de data: STRPTIME(coluna, '%m/%d/%Y') ou TRY_CAST(col AS DATE)
#   4. Remova duplicatas: QUALIFY ROW_NUMBER() OVER (...) = 1
#   5. Filtre registros inválidos (métricas negativas, datas nulas...)
#   6. Adicione colunas calculadas relevantes para o negócio

# Salve em: datalake/silver/silver_dataset.parquet
#
# log_etapa('ELT - Silver (TRANSFORM SQL)', 'OK', qtd_antes=..., qtd_depois=...)


qtd_bronze_full = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/bronze/bronze_full.parquet')
""").fetchone()[0]

conn.execute("""
COPY (
    SELECT
        ID                                          AS partida_id,
        rodata                                      AS rodada,
        STRPTIME(data, '%d/%m/%Y')::DATE            AS data,
        TRIM(hora)                                  AS hora,
        TRIM(mandante)                              AS mandante,
        TRIM(visitante)                             AS visitante,
        TRIM(vencedor)                              AS vencedor,
        TRIM(arena)                                 AS arena,
        mandante_Placar                             AS gols_mandante,
        visitante_Placar                            AS gols_visitante,
        UPPER(TRIM(mandante_Estado))                AS estado_mandante,
        UPPER(TRIM(visitante_Estado))               AS estado_visitante,
        CASE
            WHEN mandante_Placar > visitante_Placar THEN 'Vitoria Mandante'
            WHEN mandante_Placar < visitante_Placar THEN 'Vitoria Visitante'
            ELSE 'Empate'
        END                                         AS resultado,
        mandante_Placar + visitante_Placar          AS total_gols_partida
    FROM read_parquet('/content/datalake/bronze/bronze_full.parquet')
    WHERE
        TRY_STRPTIME(data, '%d/%m/%Y') IS NOT NULL
        AND YEAR(STRPTIME(data, '%d/%m/%Y')::DATE) >= 2014
        AND mandante_Placar  >= 0
        AND visitante_Placar >= 0
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ID ORDER BY ID) = 1
) TO '/content/datalake/silver/silver_full.parquet' (FORMAT PARQUET)
""")

qtd_silver_full = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/silver/silver_full.parquet')
""").fetchone()[0]

log_etapa('ELT - Silver (TRANSFORM SQL) - Partidas',
          'OK', qtd_antes=qtd_bronze_full, qtd_depois=qtd_silver_full,
          obs='Filtro 2014+, conversao de data, colunas com muitos nulos removidas, resultado e total_gols_partida adicionados')


qtd_bronze_gols = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/bronze/bronze_gols.parquet')
""").fetchone()[0]

conn.execute("""
COPY (
    SELECT
        partida_id,
        rodata                                          AS rodada,
        TRIM(clube)                                     AS clube,
        TRIM(atleta)                                    AS atleta,
        TRIM(minuto)                                    AS minuto,
        TRIM(COALESCE(tipo_de_gol, 'Normal'))           AS tipo_de_gol
    FROM read_parquet('/content/datalake/bronze/bronze_gols.parquet')
    WHERE
        partida_id IN (
            SELECT partida_id
            FROM read_parquet('/content/datalake/silver/silver_full.parquet')
        )
        AND atleta IS NOT NULL
        AND clube  IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY partida_id, clube, atleta, minuto
        ORDER BY partida_id
    ) = 1
) TO '/content/datalake/silver/silver_gols.parquet' (FORMAT PARQUET)
""")

qtd_silver_gols = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/silver/silver_gols.parquet')
""").fetchone()[0]

log_etapa('ELT - Silver (TRANSFORM SQL) - Gols',
          'OK', qtd_antes=qtd_bronze_gols, qtd_depois=qtd_silver_gols,
          obs='Filtro 2014+, tipo_de_gol nulo preenchido como Normal, TRIM aplicado')

qtd_bronze_cartoes = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/bronze/bronze_cartoes.parquet')
""").fetchone()[0]

conn.execute("""
COPY (
    SELECT
        partida_id,
        rodata                              AS rodada,
        TRIM(clube)                         AS clube,
        TRIM(cartao)                        AS cartao,
        TRIM(atleta)                        AS atleta,
        TRY_CAST(num_camisa AS INTEGER)     AS num_camisa,
        TRIM(posicao)                       AS posicao,
        TRIM(minuto)                        AS minuto
    FROM read_parquet('/content/datalake/bronze/bronze_cartoes.parquet')
    WHERE
        partida_id IN (
            SELECT partida_id
            FROM read_parquet('/content/datalake/silver/silver_full.parquet')
        )
        AND atleta IS NOT NULL
        AND clube  IS NOT NULL
        AND cartao IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY partida_id, clube, atleta, cartao, minuto
        ORDER BY partida_id
    ) = 1
) TO '/content/datalake/silver/silver_cartoes.parquet' (FORMAT PARQUET)
""")

qtd_silver_cartoes = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/silver/silver_cartoes.parquet')
""").fetchone()[0]

log_etapa('ELT - Silver (TRANSFORM SQL) - Cartoes',
          'OK', qtd_antes=qtd_bronze_cartoes, qtd_depois=qtd_silver_cartoes,
          obs='Filtro 2014+, atleta nulo removido, num_camisa convertido para INTEGER')


qtd_bronze_estat = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/bronze/bronze_estatisticas.parquet')
""").fetchone()[0]

conn.execute("""
COPY (
    SELECT
        partida_id,
        rodata                                                          AS rodada,
        TRIM(clube)                                                     AS clube,
        chutes,
        chutes_no_alvo,
        TRY_CAST(REPLACE(TRIM(posse_de_bola),   '%', '') AS DOUBLE)    AS posse_de_bola,
        passes,
        TRY_CAST(REPLACE(TRIM(precisao_passes), '%', '') AS DOUBLE)    AS precisao_passes,
        faltas,
        cartao_amarelo,
        cartao_vermelho,
        impedimentos,
        escanteios,
        cartao_amarelo + cartao_vermelho                                AS total_cartoes
    FROM read_parquet('/content/datalake/bronze/bronze_estatisticas.parquet')
    WHERE
        partida_id IN (
            SELECT partida_id
            FROM read_parquet('/content/datalake/silver/silver_full.parquet')
        )
        AND chutes         >= 0
        AND chutes_no_alvo >= 0
        AND passes         >= 0
        AND faltas         >= 0
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY partida_id, clube
        ORDER BY partida_id
    ) = 1
) TO '/content/datalake/silver/silver_estatisticas.parquet' (FORMAT PARQUET)
""")

qtd_silver_estat = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/silver/silver_estatisticas.parquet')
""").fetchone()[0]

log_etapa('ELT - Silver (TRANSFORM SQL) - Estatisticas',
          'OK', qtd_antes=qtd_bronze_estat, qtd_depois=qtd_silver_estat,
          obs='Filtro 2014+, posse_de_bola e precisao_passes convertidos para DOUBLE, total_cartoes adicionado')


[ OK  ]  ELT - Silver (TRANSFORM SQL) - Partidas        4,559 registros  (+4,606 registros)
[ OK  ]  ELT - Silver (TRANSFORM SQL) - Gols            10,820 registros
[ OK  ]  ELT - Silver (TRANSFORM SQL) - Cartoes         20,947 registros  (+6 registros)
[ OK  ]  ELT - Silver (TRANSFORM SQL) - Estatisticas    9,118 registros  (+9,212 registros)


In [ ]:
# 4.2c - Verificar a Silver
#
# Leia o Parquet e exiba: shape, schema (.dtypes) e 3 primeiras linhas
# Confirme que as transformações foram aplicadas corretamente

arquivos_silver = {
    'full'         : '/content/datalake/silver/silver_full.parquet',
    'gols'         : '/content/datalake/silver/silver_gols.parquet',
    'cartoes'      : '/content/datalake/silver/silver_cartoes.parquet',
    'estatisticas' : '/content/datalake/silver/silver_estatisticas.parquet',
}

for nome, caminho in arquivos_silver.items():
    df_check = pd.read_parquet(caminho)
    print(f"{'='*55}")
    print(f"  Silver — {nome}")
    print(f"  Shape  : {df_check.shape[0]:,} linhas x {df_check.shape[1]} colunas")
    print(f"  Dtypes :\n{df_check.dtypes}")
    print(f"  Nulos  :\n{df_check.isnull().sum()}")
    print(f"  Amostra:")
    print(df_check.head(3).to_string(index=False))
    print()


  Silver — full
  Shape  : 4,559 linhas x 14 colunas
  Dtypes :
partida_id             int64
rodada                 int64
data                  object
hora                  object
mandante              object
visitante             object
vencedor              object
arena                 object
gols_mandante          int64
gols_visitante         int64
estado_mandante       object
estado_visitante      object
resultado             object
total_gols_partida     int64
dtype: object
  Nulos  :
partida_id            0
rodada                0
data                  0
hora                  0
mandante              0
visitante             0
vencedor              0
arena                 0
gols_mandante         0
gols_visitante        0
estado_mandante       0
estado_visitante      0
resultado             0
total_gols_partida    0
dtype: int64
  Amostra:
 partida_id  rodada       data  hora    mandante   visitante    vencedor                                       arena  gols_mandante  gols_visitan

## 4.3: Camada Gold (agregações orientadas ao negócio)

> A Gold responde diretamente às perguntas de negócio definidas no início do notebook.  
> Crie uma tabela Gold para cada tema analítico que seu grupo definiu.


In [ ]:
# 4.3a - Tabela Gold 1 (defina o nome e o tema)
#
# Exemplos:
#   gold_por_categoria   - vendas e lucro agregados por categoria
#   gold_mensal          - evolução temporal de métricas
#   gold_por_regiao      - performance geográfica
#   gold_por_segmento    - análise por segmento de cliente
#
# Use SQL sobre a Silver:
#   FROM read_parquet('datalake/silver/silver_dataset.parquet')
#
# Salve em: datalake/gold/gold_tema1.parquet
#
# log_etapa('ELT - Gold (tema1)', 'OK', qtd_depois=...)

from pathlib import Path
Path('/content/datalake/gold').mkdir(parents=True, exist_ok=True)

conn.execute("""
COPY (
    SELECT
        mandante                                                          AS clube,
        COUNT(*)                                                          AS jogos_em_casa,
        SUM(CASE WHEN resultado = 'Vitoria Mandante'  THEN 1 ELSE 0 END) AS vitorias_em_casa,
        SUM(CASE WHEN resultado = 'Empate'            THEN 1 ELSE 0 END) AS empates_em_casa,
        SUM(CASE WHEN resultado = 'Vitoria Visitante' THEN 1 ELSE 0 END) AS derrotas_em_casa,
        ROUND(AVG(gols_mandante),  2)                                     AS media_gols_marcados_em_casa,
        ROUND(AVG(gols_visitante), 2)                                     AS media_gols_sofridos_em_casa,
        ROUND(
            100.0 * SUM(CASE WHEN resultado = 'Vitoria Mandante' THEN 1 ELSE 0 END)
            / COUNT(*), 2
        )                                                                 AS pct_aproveitamento_casa
    FROM read_parquet('/content/datalake/silver/silver_full.parquet')
    GROUP BY mandante
    ORDER BY pct_aproveitamento_casa DESC
) TO '/content/datalake/gold/gold_vantagem_casa.parquet' (FORMAT PARQUET)
""")

qtd_gold_1 = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/gold/gold_vantagem_casa.parquet')
""").fetchone()[0]

log_etapa('ELT - Gold (vantagem_casa)', 'OK', qtd_depois=qtd_gold_1)

conn.execute("""
    SELECT * FROM read_parquet('/content/datalake/gold/gold_vantagem_casa.parquet') LIMIT 5
""").df()


[ OK  ]  ELT - Gold (vantagem_casa)                     35 registros


,clube,jogos_em_casa,vitorias_em_casa,empates_em_casa,derrotas_em_casa,media_gols_marcados_em_casa,media_gols_sofridos_em_casa,pct_aproveitamento_casa
0,Mirassol,19,12.00,7.00,0.00,2.21,0.89,63.16
1,Palmeiras,228,142.00,45.00,41.00,1.81,0.85,62.28
2,Flamengo,228,141.00,46.00,41.00,1.80,0.85,61.84
3,Internacional,209,124.00,50.00,35.00,1.61,0.83,59.33
4,Atletico-MG,228,133.00,48.00,47.00,1.68,0.97,58.33


In [ ]:
# 4.3b - Tabela Gold 2
#
# log_etapa('ELT - Gold (tema2)', 'OK', qtd_depois=...)

conn.execute("""
COPY (
    WITH mandante AS (
        SELECT
            mandante                                                          AS clube,
            YEAR(data)                                                        AS ano,
            COUNT(*)                                                          AS jogos,
            SUM(CASE WHEN resultado = 'Vitoria Mandante'  THEN 1 ELSE 0 END) AS vitorias,
            SUM(CASE WHEN resultado = 'Empate'            THEN 1 ELSE 0 END) AS empates,
            SUM(CASE WHEN resultado = 'Vitoria Visitante' THEN 1 ELSE 0 END) AS derrotas,
            SUM(gols_mandante)                                                AS gols_marcados,
            SUM(gols_visitante)                                               AS gols_sofridos
        FROM read_parquet('/content/datalake/silver/silver_full.parquet')
        GROUP BY mandante, YEAR(data)
    ),
    visitante AS (
        SELECT
            visitante                                                         AS clube,
            YEAR(data)                                                        AS ano,
            COUNT(*)                                                          AS jogos,
            SUM(CASE WHEN resultado = 'Vitoria Visitante' THEN 1 ELSE 0 END) AS vitorias,
            SUM(CASE WHEN resultado = 'Empate'            THEN 1 ELSE 0 END) AS empates,
            SUM(CASE WHEN resultado = 'Vitoria Mandante'  THEN 1 ELSE 0 END) AS derrotas,
            SUM(gols_visitante)                                               AS gols_marcados,
            SUM(gols_mandante)                                                AS gols_sofridos
        FROM read_parquet('/content/datalake/silver/silver_full.parquet')
        GROUP BY visitante, YEAR(data)
    ),
    unificado AS (
        SELECT * FROM mandante
        UNION ALL
        SELECT * FROM visitante
    )
    SELECT
        clube,
        ano,
        SUM(jogos)          AS total_jogos,
        SUM(vitorias)       AS total_vitorias,
        SUM(empates)        AS total_empates,
        SUM(derrotas)       AS total_derrotas,
        SUM(gols_marcados)  AS total_gols_marcados,
        SUM(gols_sofridos)  AS total_gols_sofridos,
        SUM(gols_marcados) - SUM(gols_sofridos)       AS saldo_gols,
        ROUND(100.0 * SUM(vitorias) / SUM(jogos), 2) AS pct_aproveitamento
    FROM unificado
    GROUP BY clube, ano
    ORDER BY clube, ano
) TO '/content/datalake/gold/gold_desempenho_historico.parquet' (FORMAT PARQUET)
""")

qtd_gold_2 = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/gold/gold_desempenho_historico.parquet')
""").fetchone()[0]

log_etapa('ELT - Gold (desempenho_historico)', 'OK', qtd_depois=qtd_gold_2)

conn.execute("""
    SELECT * FROM read_parquet('/content/datalake/gold/gold_desempenho_historico.parquet') LIMIT 5
""").df()


[ OK  ]  ELT - Gold (desempenho_historico)              244 registros


,clube,ano,total_jogos,total_vitorias,total_empates,total_derrotas,total_gols_marcados,total_gols_sofridos,saldo_gols,pct_aproveitamento
0,America-MG,2016,38.00,7.00,7.00,24.00,23.00,58.00,-35.00,18.42
1,America-MG,2018,38.00,10.00,10.00,18.00,30.00,47.00,-17.00,26.32
2,America-MG,2021,38.00,13.00,14.00,11.00,41.00,37.00,4.00,34.21
3,America-MG,2022,38.00,15.00,8.00,15.00,40.00,40.00,0.00,39.47
4,America-MG,2023,38.00,5.00,9.00,24.00,42.00,81.00,-39.00,13.16


In [ ]:
# 4.3c - Tabela Gold 3
#
# log_etapa('ELT - Gold (tema3)', 'OK', qtd_depois=...)

conn.execute("""
COPY (
    SELECT
        clube,
        COUNT(*)                                                    AS total_gols,
        SUM(CASE WHEN tipo_de_gol = 'Normal'     THEN 1 ELSE 0 END) AS gols_normais,
        SUM(CASE WHEN tipo_de_gol = 'Penalty'    THEN 1 ELSE 0 END) AS gols_penalti,
        SUM(CASE WHEN tipo_de_gol = 'Gol Contra' THEN 1 ELSE 0 END) AS gols_contra
    FROM read_parquet('/content/datalake/silver/silver_gols.parquet')
    GROUP BY clube
    ORDER BY total_gols DESC
) TO '/content/datalake/gold/gold_gols_por_clube.parquet' (FORMAT PARQUET)
""")

qtd_gold_3 = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/gold/gold_gols_por_clube.parquet')
""").fetchone()[0]

log_etapa('ELT - Gold (gols_por_clube)', 'OK', qtd_depois=qtd_gold_3)

conn.execute("""
    SELECT * FROM read_parquet('/content/datalake/gold/gold_gols_por_clube.parquet') LIMIT 5
""").df()


[ OK  ]  ELT - Gold (gols_por_clube)                    35 registros


,clube,total_gols,gols_normais,gols_penalti,gols_contra
0,Flamengo,729,656.00,59.00,14.00
1,Palmeiras,707,637.00,56.00,14.00
2,Atletico-MG,648,555.00,72.00,21.00
3,Sao Paulo,570,512.00,51.00,7.00
4,Corinthians,566,503.00,49.00,14.00


In [ ]:
conn.execute("""
COPY (
    WITH cartoes_por_clube AS (
        SELECT
            clube,
            COUNT(*)                                                     AS total_cartoes,
            SUM(CASE WHEN cartao = 'Amarelo'  THEN 1 ELSE 0 END)        AS amarelos,
            SUM(CASE WHEN cartao = 'Vermelho' THEN 1 ELSE 0 END)        AS vermelhos
        FROM read_parquet('/content/datalake/silver/silver_cartoes.parquet')
        GROUP BY clube
    ),
    faltas_por_clube AS (
        SELECT
            clube,
            SUM(faltas)  AS total_faltas,
            COUNT(*)     AS jogos
        FROM read_parquet('/content/datalake/silver/silver_estatisticas.parquet')
        GROUP BY clube
    )
    SELECT
        c.clube,
        c.total_cartoes,
        c.amarelos,
        c.vermelhos,
        f.total_faltas,
        f.jogos,
        ROUND(c.total_cartoes * 1.0 / f.jogos, 2)  AS media_cartoes_por_jogo,
        ROUND(f.total_faltas  * 1.0 / f.jogos, 2)  AS media_faltas_por_jogo
    FROM cartoes_por_clube  c
    JOIN faltas_por_clube   f ON c.clube = f.clube
    ORDER BY total_cartoes DESC
) TO '/content/datalake/gold/gold_times_violentos.parquet' (FORMAT PARQUET)
""")

qtd_gold_4 = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/gold/gold_times_violentos.parquet')
""").fetchone()[0]

log_etapa('ELT - Gold (times_violentos)', 'OK', qtd_depois=qtd_gold_4)

conn.execute("""
    SELECT * FROM read_parquet('/content/datalake/gold/gold_times_violentos.parquet') LIMIT 5
""").df()


[ OK  ]  ELT - Gold (times_violentos)                   34 registros


,clube,total_cartoes,amarelos,vermelhos,total_faltas,jogos,media_cartoes_por_jogo,media_faltas_por_jogo
0,Fluminense,1076,1011.00,65.00,5360.00,456,2.36,11.75
1,Sao Paulo,1051,1000.00,51.00,5729.00,456,2.30,12.56
2,Internacional,1034,979.00,55.00,5223.00,418,2.47,12.50
3,Palmeiras,1033,982.00,51.00,5924.00,456,2.27,12.99
4,Atletico-MG,994,950.00,44.00,5468.00,455,2.18,12.02


In [ ]:
conn.execute("""
COPY (
    WITH gols_atleta AS (
        SELECT
            atleta,
            clube,
            COUNT(*)                                                      AS total_gols,
            SUM(CASE WHEN tipo_de_gol = 'Normal'     THEN 1 ELSE 0 END)  AS gols_normais,
            SUM(CASE WHEN tipo_de_gol = 'Penalty'    THEN 1 ELSE 0 END)  AS gols_penalti,
            SUM(CASE WHEN tipo_de_gol = 'Gol Contra' THEN 1 ELSE 0 END)  AS gols_contra
        FROM read_parquet('/content/datalake/silver/silver_gols.parquet')
        GROUP BY atleta, clube
    ),
    cartoes_atleta AS (
        SELECT
            atleta,
            clube,
            COUNT(*)                                                      AS total_cartoes,
            SUM(CASE WHEN cartao = 'Amarelo'  THEN 1 ELSE 0 END)         AS cartoes_amarelos,
            SUM(CASE WHEN cartao = 'Vermelho' THEN 1 ELSE 0 END)         AS cartoes_vermelhos
        FROM read_parquet('/content/datalake/silver/silver_cartoes.parquet')
        GROUP BY atleta, clube
    )
    SELECT
        COALESCE(g.atleta, c.atleta)            AS atleta,
        COALESCE(g.clube,  c.clube)             AS clube,
        COALESCE(g.total_gols,        0)        AS total_gols,
        COALESCE(g.gols_normais,      0)        AS gols_normais,
        COALESCE(g.gols_penalti,      0)        AS gols_penalti,
        COALESCE(g.gols_contra,       0)        AS gols_contra,
        COALESCE(c.total_cartoes,     0)        AS total_cartoes,
        COALESCE(c.cartoes_amarelos,  0)        AS cartoes_amarelos,
        COALESCE(c.cartoes_vermelhos, 0)        AS cartoes_vermelhos
    FROM gols_atleta    g
    FULL OUTER JOIN cartoes_atleta c
        ON g.atleta = c.atleta AND g.clube = c.clube
    ORDER BY total_gols DESC
) TO '/content/datalake/gold/gold_estatisticas_jogadores.parquet' (FORMAT PARQUET)
""")

qtd_gold_5 = conn.execute("""
    SELECT COUNT(*) FROM read_parquet('/content/datalake/gold/gold_estatisticas_jogadores.parquet')
""").fetchone()[0]

log_etapa('ELT - Gold (estatisticas_jogadores)', 'OK', qtd_depois=qtd_gold_5)

conn.execute("""
    SELECT * FROM read_parquet('/content/datalake/gold/gold_estatisticas_jogadores.parquet') LIMIT 5
""").df()


[ OK  ]  ELT - Gold (estatisticas_jogadores)            4,394 registros


,atleta,clube,total_gols,gols_normais,gols_penalti,gols_contra,total_cartoes,cartoes_amarelos,cartoes_vermelhos
0,Pedro,Flamengo,67,58.00,9.00,0.00,12,12.00,0.00
1,Hulk,Atletico-MG,64,46.00,18.00,0.00,24,21.00,3.00
2,Luciano da Rocha Neves,Sao Paulo,62,55.00,7.00,0.00,35,34.00,1.00
3,Gabriel Barbosa,Flamengo,60,40.00,20.00,0.00,34,29.00,5.00
4,Bruno Henrique,Flamengo,59,55.00,3.00,1.00,44,43.00,1.00


---
# Seção 5: Monitoramento

> Exibir e salvar o log completo do pipeline.  
> Se todas as etapas anteriores chamaram `log_etapa`, o relatório será gerado automaticamente.


In [ ]:
# 5.1 - Exibir o log completo do pipeline
df_log = exibir_log()
display(df_log)

,etapa,status,qtd_antes,qtd_depois,removidos,timestamp,obs
0,Extract - Leitura do dataset - (gols),OK,NaN,10820,NaN,2026-04-13T23:47:36.849043,
1,Extract - Leitura do dataset - (cartoes),OK,NaN,20953,NaN,2026-04-13T23:47:36.849597,
2,Extract - Leitura do dataset - (estatisticas),OK,NaN,18330,NaN,2026-04-13T23:47:36.849713,
3,Extract - Leitura do dataset - (full),OK,NaN,9165,NaN,2026-04-13T23:47:36.849806,
4,Staging - Salvar bruto (gols),OK,10820.00,10820,0.00,2026-04-13T23:47:37.725060,
5,Staging - Salvar bruto (cartoes),OK,20953.00,20953,0.00,2026-04-13T23:47:37.774865,
6,Staging - Salvar bruto (estatisticas),OK,18330.00,18330,0.00,2026-04-13T23:47:37.811308,
7,Staging - Salvar bruto (full),OK,9165.00,9165,0.00,2026-04-13T23:47:37.863907,
8,ETL - Filtrar partidas (2014+),OK,9165.00,4559,4606.00,2026-04-13T23:47:38.731171,
9,ETL - Filtrar gols (2014+),OK,10820.00,10820,0.00,2026-04-13T23:47:38.770833,


In [ ]:
# 5.2 - Resumo de qualidade
#
# Calcule e exiba:
#   a) Total de etapas executadas
total_etapas = df_log.shape[0]

#   b) Etapas com status OK vs. FALHA

etapas_ok = df_log[df_log.status == 'OK'].shape[0]
etapas_falha = df_log[df_log.status == 'FALHA'].shape[0]

#   c) Total de registros removidos em todo o pipeline (soma da coluna 'removidos')
total_removidos = df_log['removidos'].sum()

#   d) Percentual de dados perdidos em relação ao dataset original

total_inicial = df_log['qtd_antes'].dropna().iloc[0]

total_final = df_log['qtd_depois'].dropna().iloc[-1]

removidos = total_inicial - total_final

percentual_perda = (removidos / total_inicial) * 100

# Exemplo:
# print(f"Etapas OK    : {df_log[df_log.status=='OK'].shape[0]}")
# print(f"Etapas FALHA : {df_log[df_log.status=='FALHA'].shape[0]}")

print(f"Total de etapas         : {total_etapas}")
print(f"Etapas OK              : {etapas_ok}")
print(f"Etapas FALHA           : {etapas_falha}")
print(f"Total removidos        : {removidos}")
print(f"Percentual de perda    : {percentual_perda:.2f}%")


Total de etapas         : 37
Etapas OK              : 37
Etapas FALHA           : 0
Total removidos        : 6426.0
Percentual de perda    : 59.39%


In [ ]:
# 5.3 - Salvar o log em JSON
salvar_log('logs/pipeline_log.json')

Log salvo em: logs/pipeline_log.json


In [ ]:
# 5.4 - Validação cruzada: DW vs. Data Lake
#
# Compare os totais entre o DW e a camada Gold:
#   a) A soma da métrica principal na fato do DW bate com a Gold?
resultado = conn.execute("""show tables""").fetchone()[0]

print(resultado)
#   b) O número de registros da Silver bate com a fato do DW (após tratamentos)?
#
# Se houver divergência, investigue e documente


dim_atleta


---
# Seção 6: Analytics (OLAP)

> Execute as 5 queries que respondem às perguntas de negócio definidas no início.  
> Cada query deve ser executada **no DW** (com JOIN nas dimensões) **e também na camada Gold** (comparando os resultados).


In [ ]:
# Query 1 - (Qual a vantagem de jogar em casa? (aproveitamento, gols marcados e sofridos))
#
# Responder no DW (use JOIN com pelo menos uma dimensão):

conn = duckdb.connect('dw_projeto.duckdb')

# A fato_eventos contém gols; para resultado de partida usamos a Silver (não há fato de partidas no DW)
# Usamos a Gold que foi construída a partir da Silver
conn.execute("""
    SELECT *
    FROM read_parquet('/content/datalake/gold/gold_vantagem_casa.parquet')
    ORDER BY pct_aproveitamento_casa DESC
    LIMIT 10
""").df()


,clube,jogos_em_casa,vitorias_em_casa,empates_em_casa,derrotas_em_casa,media_gols_marcados_em_casa,media_gols_sofridos_em_casa,pct_aproveitamento_casa
0,Mirassol,19,12.00,7.00,0.00,2.21,0.89,63.16
1,Palmeiras,228,142.00,45.00,41.00,1.81,0.85,62.28
2,Flamengo,228,141.00,46.00,41.00,1.80,0.85,61.84
3,Internacional,209,124.00,50.00,35.00,1.61,0.83,59.33
4,Atletico-MG,228,133.00,48.00,47.00,1.68,0.97,58.33
5,Gremio,209,121.00,44.00,44.00,1.57,0.89,57.89
6,Santos,209,115.00,53.00,41.00,1.62,0.87,55.02
7,Ponte Preta,57,31.00,9.00,17.00,1.42,0.95,54.39
8,Athletico-PR,209,113.00,52.00,44.00,1.49,0.84,54.07
9,Corinthians,228,123.00,75.00,30.00,1.51,0.82,53.95


In [ ]:
# Query 1 - mesma pergunta, respondida na camada Gold:
# FROM read_parquet('datalake/gold/gold_tema_X.parquet')

conn.execute("""
    SELECT
        clube,
        jogos_em_casa,
        vitorias_em_casa,
        empates_em_casa,
        derrotas_em_casa,
        media_gols_marcados_em_casa,
        media_gols_sofridos_em_casa,
        pct_aproveitamento_casa
    FROM read_parquet('/content/datalake/gold/gold_vantagem_casa.parquet')
    ORDER BY pct_aproveitamento_casa DESC
    LIMIT 10
""").df()


,clube,jogos_em_casa,vitorias_em_casa,empates_em_casa,derrotas_em_casa,media_gols_marcados_em_casa,media_gols_sofridos_em_casa,pct_aproveitamento_casa
0,Mirassol,19,12.00,7.00,0.00,2.21,0.89,63.16
1,Palmeiras,228,142.00,45.00,41.00,1.81,0.85,62.28
2,Flamengo,228,141.00,46.00,41.00,1.80,0.85,61.84
3,Internacional,209,124.00,50.00,35.00,1.61,0.83,59.33
4,Atletico-MG,228,133.00,48.00,47.00,1.68,0.97,58.33
5,Gremio,209,121.00,44.00,44.00,1.57,0.89,57.89
6,Santos,209,115.00,53.00,41.00,1.62,0.87,55.02
7,Ponte Preta,57,31.00,9.00,17.00,1.42,0.95,54.39
8,Athletico-PR,209,113.00,52.00,44.00,1.49,0.84,54.07
9,Corinthians,228,123.00,75.00,30.00,1.51,0.82,53.95


In [ ]:
# Query 2 - (Desempenho histórico dos times por ano)
#
# No DW: junta fato_eventos com dim_clube e dim_data

conn.execute("""
    SELECT
        dc.clube,
        dd.ano,
        SUM(fe.qtd_gol)      AS total_gols_marcados,
        COUNT(DISTINCT fe.partida_id) AS partidas_com_gol
    FROM fato_eventos   fe
    JOIN dim_clube      dc ON fe.sk_clube  = dc.sk_clube
    JOIN dim_data       dd ON fe.sk_data   = dd.sk_data
    GROUP BY dc.clube, dd.ano
    ORDER BY dc.clube, dd.ano
    LIMIT 20
""").df()


,clube,ano,total_gols_marcados,partidas_com_gol
0,America-Mg,2016,23.00,36
1,America-Mg,2018,30.00,36
2,America-Mg,2021,41.00,37
3,America-Mg,2022,40.00,36
4,America-Mg,2023,42.00,37
5,Athletico-Pr,2014,43.00,37
6,Athletico-Pr,2015,43.00,38
7,Athletico-Pr,2016,38.00,37
8,Athletico-Pr,2017,45.00,38
9,Athletico-Pr,2018,54.00,38


In [ ]:
# Query 2 - na camada Gold: Desempenho histórico na camada Gold (mais completo: V/E/D/aproveitamento)

conn.execute("""
    SELECT *
    FROM read_parquet('/content/datalake/gold/gold_desempenho_historico.parquet')
    ORDER BY clube, ano
    LIMIT 20
""").df()


,clube,ano,total_jogos,total_vitorias,total_empates,total_derrotas,total_gols_marcados,total_gols_sofridos,saldo_gols,pct_aproveitamento
0,America-MG,2016,38.00,7.00,7.00,24.00,23.00,58.00,-35.00,18.42
1,America-MG,2018,38.00,10.00,10.00,18.00,30.00,47.00,-17.00,26.32
2,America-MG,2021,38.00,13.00,14.00,11.00,41.00,37.00,4.00,34.21
3,America-MG,2022,38.00,15.00,8.00,15.00,40.00,40.00,0.00,39.47
4,America-MG,2023,38.00,5.00,9.00,24.00,42.00,81.00,-39.00,13.16
5,Athletico-PR,2014,38.00,15.00,9.00,14.00,43.00,42.00,1.00,39.47
6,Athletico-PR,2015,38.00,14.00,9.00,15.00,43.00,48.00,-5.00,36.84
7,Athletico-PR,2016,38.00,17.00,6.00,15.00,38.00,32.00,6.00,44.74
8,Athletico-PR,2017,38.00,14.00,9.00,15.00,45.00,43.00,2.00,36.84
9,Athletico-PR,2018,38.00,16.00,9.00,13.00,54.00,37.00,17.00,42.11


In [ ]:
# Query 3 - Quais times fizeram mais gols?
#
# No DW: agrega gols por clube via fato_eventos + dim_clube
import duckdb
conn = duckdb.connect('dw_projeto.duckdb')
conn.execute("""
    SELECT
        dc.clube,
        SUM(fe.qtd_gol)                                             AS total_gols,
        SUM(CASE WHEN dtg.tipo_de_gol = 'Normal'     THEN 1 ELSE 0 END) AS gols_normais,
        SUM(CASE WHEN dtg.tipo_de_gol = 'Penalty'    THEN 1 ELSE 0 END) AS gols_penalti,
        SUM(CASE WHEN dtg.tipo_de_gol = 'Gol Contra' THEN 1 ELSE 0 END) AS gols_contra
    FROM fato_eventos   fe
    JOIN dim_clube      dc  ON fe.sk_clube    = dc.sk_clube
    JOIN dim_tipo_gol   dtg ON fe.sk_tipo_gol = dtg.sk_tipo_gol
    GROUP BY dc.clube
    ORDER BY total_gols DESC
    LIMIT 10
""").df()


,clube,total_gols,gols_normais,gols_penalti,gols_contra
0,Flamengo,729.00,656.00,59.00,14.00
1,Palmeiras,707.00,637.00,56.00,14.00
2,Atletico-Mg,648.00,555.00,72.00,21.00
3,Sao Paulo,570.00,512.00,51.00,7.00
4,Corinthians,566.00,503.00,49.00,14.00
5,Fluminense,556.00,479.00,51.00,26.00
6,Gremio,547.00,482.00,51.00,14.00
7,Internacional,528.00,464.00,54.00,10.00
8,Santos,523.00,452.00,52.00,19.00
9,Athletico-Pr,492.00,443.00,35.00,14.00


In [ ]:
# Query 3 - na camada Gold: Times com mais gols na camada Gold


conn.execute("""
    SELECT *
    FROM read_parquet('/content/datalake/gold/gold_gols_por_clube.parquet')
    LIMIT 10
""").df()

,clube,total_gols,gols_normais,gols_penalti,gols_contra
0,Flamengo,729,656.00,59.00,14.00
1,Palmeiras,707,637.00,56.00,14.00
2,Atletico-MG,648,555.00,72.00,21.00
3,Sao Paulo,570,512.00,51.00,7.00
4,Corinthians,566,503.00,49.00,14.00
5,Fluminense,556,479.00,51.00,26.00
6,Gremio,547,482.00,51.00,14.00
7,Internacional,528,464.00,54.00,10.00
8,Santos,523,452.00,52.00,19.00
9,Athletico-PR,492,443.00,35.00,14.00


In [ ]:
# Query 4 - Times mais violentos (mais cartões)
#
# No DW: fato_eventos não contém cartões diretamente (só gols)
# Usamos a silver_cartoes via DuckDB para manter a consulta no ecossistema
conn.execute("""
    SELECT
        clube,
        COUNT(*)                                                       AS total_cartoes,
        SUM(CASE WHEN cartao = 'Amarelo'  THEN 1 ELSE 0 END)          AS amarelos,
        SUM(CASE WHEN cartao = 'Vermelho' THEN 1 ELSE 0 END)          AS vermelhos
    FROM read_parquet('/content/datalake/silver/silver_cartoes.parquet')
    GROUP BY clube
    ORDER BY total_cartoes DESC
    LIMIT 10
""").df()


,clube,total_cartoes,amarelos,vermelhos
0,Fluminense,1076,1011.00,65.00
1,Sao Paulo,1051,1000.00,51.00
2,Internacional,1034,979.00,55.00
3,Palmeiras,1033,982.00,51.00
4,Atletico-MG,994,950.00,44.00
5,Athletico-PR,977,938.00,39.00
6,Santos,950,898.00,52.00
7,Flamengo,933,884.00,49.00
8,Gremio,933,889.00,44.00
9,Corinthians,905,866.00,39.00


In [ ]:
# Query 4 - Times mais violentos na camada Gold (inclui faltas e médias)

conn.execute("""
    SELECT *
    FROM read_parquet('/content/datalake/gold/gold_times_violentos.parquet')
    LIMIT 10
""").df()

,clube,total_cartoes,amarelos,vermelhos,total_faltas,jogos,media_cartoes_por_jogo,media_faltas_por_jogo
0,Fluminense,1076,1011.00,65.00,5360.00,456,2.36,11.75
1,Sao Paulo,1051,1000.00,51.00,5729.00,456,2.30,12.56
2,Internacional,1034,979.00,55.00,5223.00,418,2.47,12.50
3,Palmeiras,1033,982.00,51.00,5924.00,456,2.27,12.99
4,Atletico-MG,994,950.00,44.00,5468.00,455,2.18,12.02
5,Athletico-PR,977,938.00,39.00,4875.00,418,2.34,11.66
6,Santos,950,898.00,52.00,5430.00,418,2.27,12.99
7,Flamengo,933,884.00,49.00,5502.00,456,2.05,12.07
8,Gremio,933,889.00,44.00,4756.00,418,2.23,11.38
9,Corinthians,905,866.00,39.00,4991.00,456,1.98,10.95


In [ ]:
# Query 5 - Estatísticas de jogadores (gols e cartões)
#
# No DW: fato_eventos + dim_atleta + dim_tipo_gol
conn.execute("""
  SELECT
    da.atleta,
    dc.clube,
    SUM(fe.qtd_gol) AS total_gols,
    SUM(CASE WHEN dtg.tipo_de_gol = 'Normal' THEN fe.qtd_gol ELSE 0 END) AS gols_normais,
    SUM(CASE WHEN dtg.tipo_de_gol = 'Penalty' THEN fe.qtd_gol ELSE 0 END) AS gols_penalti,
    SUM(CASE WHEN dtg.tipo_de_gol = 'Gol Contra' THEN fe.qtd_gol ELSE 0 END) AS gols_contra,
    SUM(fe.qtd_cartao) AS total_cartoes,
    SUM(CASE WHEN dtc.cartao = 'Amarelo' THEN fe.qtd_cartao ELSE 0 END) AS cartoes_amarelos,
    SUM(CASE WHEN dtc.cartao = 'Vermelho' THEN fe.qtd_cartao ELSE 0 END) AS cartoes_vermelhos
FROM fato_eventos fe
JOIN dim_atleta da
    ON fe.sk_atleta = da.sk_atleta
JOIN dim_clube dc
    ON fe.sk_clube = dc.sk_clube
LEFT JOIN dim_tipo_gol dtg
    ON fe.sk_tipo_gol = dtg.sk_tipo_gol
LEFT JOIN dim_cartao dtc
    ON fe.sk_cartao = dtc.sk_cartao
GROUP BY da.atleta, dc.clube
ORDER BY total_gols DESC, total_cartoes DESC, da.atleta
LIMIT 10
""").df()

,atleta,clube,total_gols,gols_normais,gols_penalti,gols_contra,total_cartoes,cartoes_amarelos,cartoes_vermelhos
0,Pedro,Flamengo,67.00,58.00,9.00,0.00,12.00,12.00,0.00
1,Hulk,Atletico-Mg,64.00,46.00,18.00,0.00,24.00,21.00,3.00
2,Luciano Da Rocha Neves,Sao Paulo,62.00,55.00,7.00,0.00,35.00,34.00,1.00
3,Gabriel Barbosa,Flamengo,60.00,40.00,20.00,0.00,34.00,29.00,5.00
4,Bruno Henrique,Flamengo,59.00,55.00,3.00,1.00,44.00,43.00,1.00
5,Giorgian De Arrascaeta,Flamengo,55.00,52.00,3.00,0.00,11.00,11.00,0.00
6,Eduardo Pereira Rodrigues,Palmeiras,53.00,53.00,0.00,0.00,38.00,36.00,2.00
7,Gilberto Oliveira Souza Junior,Bahia,46.00,36.00,10.00,0.00,18.00,18.00,0.00
8,Germán Cano,Fluminense,46.00,45.00,1.00,0.00,17.00,16.00,1.00
9,Gabriel Barbosa,Santos,41.00,37.00,4.00,0.00,20.00,19.00,1.00


In [ ]:
# Query 5 - na camada Gold: Estatísticas de jogadores na camada Gold (gols + cartões combinados)

conn.execute("""
    SELECT *
    FROM read_parquet('/content/datalake/gold/gold_estatisticas_jogadores.parquet')
    LIMIT 10
""").df()

,atleta,clube,total_gols,gols_normais,gols_penalti,gols_contra,total_cartoes,cartoes_amarelos,cartoes_vermelhos
0,Pedro,Flamengo,67,58.00,9.00,0.00,12,12.00,0.00
1,Hulk,Atletico-MG,64,46.00,18.00,0.00,24,21.00,3.00
2,Luciano da Rocha Neves,Sao Paulo,62,55.00,7.00,0.00,35,34.00,1.00
3,Gabriel Barbosa,Flamengo,60,40.00,20.00,0.00,34,29.00,5.00
4,Bruno Henrique,Flamengo,59,55.00,3.00,1.00,44,43.00,1.00
5,Eduardo Pereira Rodrigues,Palmeiras,53,53.00,0.00,0.00,38,36.00,2.00
6,Gilberto Oliveira Souza Junior,Bahia,46,36.00,10.00,0.00,18,18.00,0.00
7,Germán Cano,Fluminense,46,45.00,1.00,0.00,17,16.00,1.00
8,Yuri Alberto,Corinthians,41,36.00,5.00,0.00,17,15.00,2.00
9,Gabriel Barbosa,Santos,41,37.00,4.00,0.00,20,19.00,1.00


---
# Seção 7: Conclusão

Responda as questões abaixo com base na experiência de implementar este projeto.

---

## 7.1 - ETL vs. ELT no contexto do seu dataset

**a) Qual abordagem foi mais trabalhosa de implementar? Por quê?**  

*O ETL foi mais trabalhoso. Precisamos escrever toda a lógica em Python: tratar nulos, converter tipos, montar as dimensões com surrogate keys e fazer os mapeamentos para a tabela fato. Qualquer erro em um passo quebrava os seguintes. No ELT, tudo ficou em queries SQL mais diretas e fáceis de ajustar individualmente.*


**b) As queries no DW (com JOIN nas dimensões) e na camada Gold retornaram os mesmos resultados?  
Se houve divergência, o que a causou?**  

*Sim, os resultados foram equivalentes. A pequena diferença esperada é que a fato_eventos guarda só gols, então partidas sem gol (0×0) não aparecem nela. A Gold foi construída a partir da Silver que tem todas as partidas, então é mais completa para métricas de resultado. Para contagem de gols os dois batem.*

**c) Se os dados da fonte forem atualizados (novos registros chegarem amanhã),  
qual pipeline seria mais fácil de reexecutar - ETL ou ELT? Justifique.**  

*O ELT. Basta jogar os novos arquivos na camada Bronze e rodar as queries SQL da Silver e Gold novamente. No ETL seria necessário reprocessar toda a cadeia Python e reinserir no DuckDB, tomando cuidado para não duplicar registros.*

---

## 7.2 - Recomendação arquitetural

**Se este projeto fosse colocado em produção para uma empresa real,  
qual arquitetura o grupo recomendaria?**

Considere:
- Volume dos dados e frequência de atualização
- Perfil dos usuários (analistas de negócio, cientistas de dados, engenheiros)
- Necessidade de auditoria e rastreabilidade
- Custo de manutenção do pipeline

*Para produção, recomendaríamos manter a arquitetura ELT com Data Lake Medallion (Bronze → Silver → Gold). O Campeonato Brasileiro tem volume previsível (380 partidas por ano), então o reprocessamento incremental é simples e barato. Analistas de negócio consumiriam a camada Gold via Power BI ou Metabase, enquanto cientistas de dados trabalhariam diretamente na Silver em Parquet. O pipeline poderia ser orquestrado com Airflow ou Prefect para rodar automaticamente após cada rodada. A rastreabilidade já está garantida pelo log implementado. O custo de manutenção seria baixo porque toda a lógica está em SQL versionado.*

---

## 7.3 - O que vocês fariam diferente

**Cite 2 decisões que tomariam de forma diferente se recomeçassem o projeto hoje.**

1. *Criaríamos uma fato_partidas separada no DW para armazenar resultado, placar e árbitro de cada jogo — assim as perguntas sobre vitórias e aproveitamento poderiam ser respondidas diretamente no DW sem depender da camada Silver.*

2. *Faríamos uma etapa de padronização de nomes de clubes logo no início (por exemplo, um dicionário mapeando "Atletico MG", "Atlético Mineiro" e "Atletico-MG" para um nome canônico único), evitando divergências nas análises históricas.*


In [ ]:
# Encerramento - fechar conexões e exibir resumo final

try:
    conn.close()
except:
    pass

try:
    conn_lake.close()
except:
    pass

print('\n' + '='*60)
print('  PROJETO FINAL - RESUMO DE EXECUÇÃO')
print('='*60)

df_log = exibir_log()
if df_log is not None:
    ok    = df_log[df_log['status'] == 'OK'].shape[0]
    falha = df_log[df_log['status'] == 'FALHA'].shape[0]
    print(f'Etapas executadas : {len(df_log)}')
    print(f'OK                : {ok}')
    print(f'FALHA             : {falha}')
    removidos = df_log['removidos'].sum()
    print(f'Registros removidos no pipeline: {removidos:,.0f}')

print('='*60)
salvar_log('logs/pipeline_log.json')


  PROJETO FINAL - RESUMO DE EXECUÇÃO
Etapas executadas : 37
OK                : 37
FALHA             : 0
Registros removidos no pipeline: 27,642
Log salvo em: logs/pipeline_log.json
